# Metref
- # based on https://gitlab.com/helpdesk.landsaf/lsasaf_data_access/-/tree/main/examples/thredds

In [ ]:
#this was implemented in colab. to clone the gits in pc, powershell. 
import os
import sys
import datetime as dt
import pandas as pd
import certifi
from concurrent.futures import ThreadPoolExecutor
!rm -rf /content/lsasaf_data_access/ 
!git clone https://gitlab.com/helpdesk.landsaf/lsasaf_data_access.git
import sys
sys.path.insert(0,r'/content/lsasaf_data_access/examples/thredds/')
# Required to load netcdf files using OpenDAP (valid in May 2023)
!pip install 'netCDF4<1.6'
import netCDF4
import datetime as dt
import thredds_lsasaf_utils as tlu
server_user = ''
server_passwd = ''

In [ ]:


product_path= r"/MSG/METREF/NETCDF/"
product_fname="NETCDF4_LSASAF_MSG_METREF_MSG-Disk"
product_freq="d"
NcvarsLoad=['METREF']
LatLonBox = [47, 55, 5.5, 15] # Germany
main_dir = r"/content/drive/MyDrive/cerra/ls"

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Initialize product object
product = tlu.lsa_product(product_path, product_fname)
product.user = server_user
product.passwd = server_passwd

def merge_all(save_dir, main_dir, year):
    '''
    data downloaded daily, and merged into a year at last. 
    helper funciton for process_year()
    '''
  save_name = os.path.join(main_dir, f'{year}.nc')
  nc_files = sorted([os.path.join(save_dir, f)
                   for f in os.listdir(save_dir) if f.endswith(".nc")])
  ds_all = xr.open_mfdataset(
    nc_files,
    combine="by_coords",  
    parallel=True,        
    engine="netcdf4"      
    )
  ds_all = ds_all.sortby("time")
  ds_all.to_netcdf(save_name)
  ds_all.close()

def process_year(year):
    '''
    
    This extracts netcdf for 1 day at a time for a year. 
    The original base code provides feature to download from certain date to another date, 
        but while using it, the date-time stamp were not accurate. All time-stamps oscillated around a date by few seconds.
    Then, merges all ncs for a year into single netcdf. 
    
    '''
    save_dir = os.path.join(main_dir,str(year))
    os.makedirs(save_dir, exist_ok = True)
    for month in range(1,13):
        days = calendar.monthrange(year, month)[1]
    for day in range(1,days+1):

        dstart = dt.datetime(year, month, day,0,0)
        dend = dt.datetime(year, month, day,0,0) #data downloaded one day at a time, date-time stamp error
        print(f'Starting Year: {dstart}')
      # dend = dt.datetime(year, 12, 31, 0, 0)
        out_file = os.path.join(save_dir, f"{dstart.date()}.nc")
        if os.path.exists(out_file):
            print(out_file)
            # print(f"Skipping {year}, file already exists.")
            continue
        else:
            try:

                slot_list = tlu.gen_slot_list(dstart, dend, product_freq)

                ds_year = tlu.load_product_slots_domain(
                    product,
                    slot_list,
                    NcvarsLoad,
                    LatLonBox=LatLonBox
                  )

                ds_year.to_netcdf(out_file)
                print(f"Successfully saved {dstart}")


            except Exception as e:
                print(f"Error processing {year}: {e}")
    merge_all(save_dir, main_dir, year)

years_to_process = list(range(2004, 2005))

for year_todo in range(2004, 2023):
    process_year(year_todo)
print("All tasks complete.")

# ERA5Land

## downloading ERA5_Land

In [ ]:
#datasets were downloaded in chunks and merged later
import cdsapi
import os
client = cdsapi.Client()
dataset = "reanalysis-era5-land"
base_path = "/content/drive/MyDrive/pcp_pet_era5_land"
years = range(2001, 2027)
all_months = [f"{m:02d}" for m in range(1, 13)]
days = [f"{d:02d}" for d in range(1, 32)]
times = [f"{h:02d}:00" for h in range(24)]
# Split months into chunks of 3
month_chunks = [all_months[i:i+3] for i in range(0, 12, 3)]
for year in years:
    for chunk in month_chunks:
        filename = f"/content/drive/MyDrive/era5land_{year}_{chunk[0]}-{chunk[-1]}.zip"
        if os.path.exists(filename):
            print(f"Skipping {filename} (already exists)")
            continue
        print(f"Downloading {year}, months {chunk}...")
        request = {
            "variable": [
                "potential_evaporation",
                "total_precipitation"
            ],
            "year": str(year),
            "month": chunk,
            "day": days,
            "time": times,
            "area": [55, 5.5, 47, 15],  # Germany
            "format": "netcdf"
        }

        client.retrieve(dataset, request).download(filename)
        print(f"Saved: {filename}")

In [ ]:
#extracting from zip if needed
import zipfile
import pathlib
import shutil
folder = pathlib.Path(r".\PET\pcp_pet_era5_land")
for zip_path in folder.glob("*.zip"):
    # Extract
    extract_dir = folder / zip_path.stem
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # Move contents up
    for item in extract_dir.iterdir():
        if item.is_file():
            new_name = folder / f"{extract_dir.name}_{item.name}"
            shutil.move(item, new_name)

    # Remove empty folder
    extract_dir.rmdir()

## creating hourly pet from raw dataset

In [ ]:
def get_hourly_erad_land(main_dir, save_dir, 
                         years = range(2001, 2002)):
    '''
    ERA5_Land accumulates pet by 24 hours. each day pet is reset at 01:00:00 hour. 
    Hourly pet created by lagging the dataset by an hour, and subtracting original dataset by lagged ds. 
    For hourly dataset of a year, previous year dataset also needs to be read for year-01-01-00:00:00. 
    '''
    
    for year in years:
        this_year = os.path.join(main_dir,f'era5land_{year}.nc')
        last_year = os.path.join(main_dir,f'era5land_{year-1}.nc')
        save_file = os.path.join(save_dir,f'era5land_{year}.nc')
        print(last_year, this_year)
        ds0 = xr.open_dataset(last_year)
        ds1 = xr.open_dataset(this_year)
        ds_comb = xr.concat([ds0.isel(valid_time=slice(-1, None)), ds1],
                            dim="valid_time")
        da = ds_comb['pev'].shift(valid_time = 1) #lagged
        db = ds_comb['pev'] - da #differenced
        condition = (ds_comb['valid_time'].dt.hour == 1).to_numpy() #reset at ever hour == 1
        condition_3d = np.broadcast_to(condition[:,np.newaxis, np.newaxis], 
                                       (ds_comb.valid_time.shape[0], 
                                        ds_comb.latitude.shape[0], 
                                        ds_comb.longitude.shape[0]))
        hourly = xr.where(condition_3d, ds_comb, db) #if hour == 1 keep original, else lagged
        hourly_trimmed = hourly.drop_isel(valid_time=0) #left out 1 time step from previous year
        hourly_trimmed.to_netcdf(save_file, format = 'NETCDF4')
        ds0.close(), ds1.close(), ds_comb.close(), hourly.close(), hourly_trimmed.close()
        # break
    # return hourly_trimmed 
get_hourly_erad_land(main_dir = r'.\pet\era5_land\annual',
                     save_dir = r'.\pet\era5_land\annual_hourly')

# CERRA

## downloading cerra

In [ ]:
import cdsapi
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor
client = cdsapi.Client()
base_path = r".\pet\cerra"
dataset = "reanalysis-cerra-single-levels"
years = range(2001, 2023)
all_months = [f"{m:02d}" for m in range(1, 13)]
days = [f"{d:02d}" for d in range(1, 32)]

client = cdsapi.Client()
base_path = r"G:\WATENV\PET\cerra"
dataset = "reanalysis-cerra-single-levels"
years = range(2001, 2023)
all_months = [f"{m:02d}" for m in range(1, 13)]
days = [f"{d:02d}" for d in range(1, 32)]
times = [f"{h:02d}:00" for h in range(24)]
# Split months into chunks of 3
month_chunks = [all_months[i:i+3] for i in range(0, 12, 3)] #split into 3 months chunk, as >3 months exceeds cds limits
for year in years:
    for chunk in month_chunks:
        file = f"cerra_{year}_{chunk[0]}-{chunk[-1]}.zip"
        filename = os.path.join(base_path, file)
        if os.path.exists(filename):
            print(f"Skipping {filename} (already exists)")
            continue
        print(f"Downloading {year}, months {chunk}...")
        request = {
        "variable": ["evaporation"],
        "level_type": "surface_or_atmosphere",
        "data_type": ["reanalysis"],
        "product_type": "forecast",
        "year": str(year),
            "month": chunk,
            "day": days,
            # "area": [55, 5.5, 47, 15],  # Germany #extraction by area not possible in CERRA
            "format": "netcdf",
            "time": [
            "00:00", "03:00", "06:00",
            "09:00", "12:00", "15:00",
            "18:00", "21:00"
        ],
        "leadtime_hour": [
            "1",
            "2",
            "3"
        ],
        "data_format": "netcdf"
    }
        client.retrieve(dataset, request).download(filename)
        print(f"Saved: {filename}")

## creating hourly dataset

In [ ]:
def get_lagged(main_dir, save_path):
    '''
    CERRA provides pet accumulated from x+1 hour to next x+3 hours
        dataset should be lagged accordingly, such that x+1 values are kept. 
    '''
    
    
    files = [(os.path.join(main_dir, file), os.path.basename(file).split('.')[0]) 
             for file in os.listdir(main_dir)]
    for file_path ,year in files:
        print(year, file_path)
        save_file = os.path.join(save_path, f'{year}.nc')
        ds = xr.open_dataset(file_path)
        da = ds['eva'].shift(valid_time = 1) #lagged
        db = ds['eva'] - da #differenced
        time_indices = np.arange(len(ds.valid_time)) #equivalent to is_reset = (ds['valid_time'].dt.hour % 4 == 0)
        condition = (time_indices % 3 == 0)
        condition_3d = np.broadcast_to(condition[:,np.newaxis, np.newaxis], (ds.valid_time.shape[0], ds.y.shape[0], ds.x.shape[0]))
        
        hourly = xr.where(condition_3d, ds, db) #replaced when forecast is reset
        hourly.eva[:24,0,0].plot()
        hourly.to_netcdf(save_file, format = 'NETCDF4')
        hourly.close()
        ds.close()
        da.close()
get_lagged(main_dir = r'.\pet\cerra\monthly\annual',
          save_path = r'.\pet\cerra\monthly\annual_lagged')

## geocoding cerra

In [51]:
from pyproj import Transformer
import rioxarray
def geocode_ds(ds):
    resolution = 5500 #resolution of cerra dataset
    geo_string = f"+proj=longlat +a={6371229} +b={6371229} +no_defs"#sphere used in lambert
    proj_string = (
        "+proj=lcc +lat_1=50 +lat_2=50 +lat_0=50 +lon_0=8 "
        "+x_0=0 +y_0=0 +a=6371229 +b=6371229 +units=m +no_defs"
        )   #projection metadata from cerra; contains sphere radius, northing easting origin 
    
    ds.rio.write_crs(proj_string, inplace=True)
    # lcp = pyproj.CRS(proj_string)
    trans_geo_to_lcc = Transformer.from_crs(geo_string, proj_string, always_xy=True, accuracy=0.0) #transformer object
    
    easting0, northing0 = trans_geo_to_lcc.transform(ds.longitude.values[0,0], ds.latitude.values[0,0]) #northing easting of first point
    # easting0, northing0
    ds['x'] = [easting0+i*resolution for i      #all grids are equidistant, so added resolution to the easting of the origin
            in range(ds.x.shape[0])]            #whil
    ds['y'] = [northing0+i*resolution for i 
            in range(ds.y.shape[0])]
    ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y")   #dataset is geocoded now
    
    return ds

# hpet dataset

## downloading hpet
    - written with copilot
    - the paper provides a .py file to download a selected region from hPET dataset, 
        but that doesn't work, required Bristol University HPC id. access denied.
    - downloads 6 hpet files at once. 1 file ~ 55 GB, so this takes time. 

In [ ]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

# URL of the dataset page
URL = "https://data.bris.ac.uk/data/dataset/qb8ujazzda0s2aykkv0oq0ctp"

# Directory to save files
SAVE_DIR = r"D:\hpet global"
os.makedirs(SAVE_DIR, exist_ok=True)

# Regex to capture year
pattern = re.compile(r"(\d{4})_hourly_pet\.nc$")

# Fetch page
response = requests.get(URL)
soup = BeautifulSoup(response.text, "html.parser")
links = soup.find_all("a")

# Collect matching file URLs
file_urls = []
for link in links:
    href = link.get("href")
    if href:
        match = pattern.search(href)
        if match:
            year = int(match.group(1))
            if 2001 <= year <= 2022:
                if href.startswith("/"):
                    href = "https://data.bris.ac.uk" + href
                file_urls.append((year, href))
file_urls = list(set(file_urls))
file_urls.sort()
                 
print(f"Found {len(file_urls)} files to download:    ", file_urls)
# --- Download function with retry ---
def download_file(url):
    filename = url.split("/")[-1]
    save_path = os.path.join(SAVE_DIR, filename)

    for attempt in range(5):
        try:
            start_time = time.time()
            with requests.get(url, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(save_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
            end_time = time.time()  # ⏱ end timer
            total_time = end_time - start_time
            return f"Downloaded {filename} in {total_time:.2f} seconds"
        except Exception as e:
            time.sleep(3)
            print(f"Retry {attempt+1}/5 for {filename} due to: {e}")

    return f"Failed to download {filename}"

# --- Parallel download: 6 at a time ---
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(download_file, url) for (_, url) in file_urls]

    for future in as_completed(futures):
        print(future.result())
print("All downloads completed.")

## clipping hpet 
    - code for clipping is copied from https://doi.org/10.5523/bris.qb8ujazzda0s2aykkv0oq0ctp provided with dataset. 
    - extents changed for germany. 
    - had an if else statement error in original, fixed here.

In [ ]:
import numpy as np
from netCDF4 import Dataset
import sys
import datetime as dt
def main():
    # example (please change these values to your specification)
    # input arguments
    # what do you want to extract single point(0) or a region(1)?
    spatial_res = 1

    # what is the period?
    startyear = 2003
    endyear = 2022

    # what is the extent of the region?
    # this only be used if spatial_res = 1
    latmin = 46#5.5
    latmax =  56#15
    lonmin = 4.5#47
    lonmax = 16#55

    # what is the location of your single point?
    # this only be used if spatial_res = 0
    latval = 3.8 
    lonval = 36.6 
 
    # What is the specific naming you want to use for the area?
    regionname= 'germany'

    # what is the time resolution hPET you want hourly or daily? 
    t_resolution ='hourly'

    # what is the directory where you download and put the hPET data?
    # download the data from the Birstol repository manually and save it 
    # in your local machine.
    data_path = 'D:/hpet global/'

    # where do you want the extracted subset of the data to be?
    output_path = 'D:/hpet global/germany/'

    # run script
    if spatial_res == 0:
        wrapper_singlepoint(startyear,endyear,latval,lonval,regionname,t_resolution,data_path,output_path)
    elif spatial_res == 1:
        wrapper_region(startyear,endyear,latmin,latmax,lonmin,lonmax,regionname,t_resolution,data_path,output_path)
    else:
        raise ValueError('spatial_res only takes 0 or 1 please check!!')


## ********** NO CHANGE ON THE CODE BELOW THIS **********************************************##
def wrapper_region(startyear,endyear,latmin,latmax,lonmin,lonmax,regionname,t_resolution,data_path,output_path):
    """
    This is a wrapper function to run for downloading hPET and dPET data.
    All the arguments need to be given at the end of the script before running
    the script.

    :param startyear: the begging year to start the data download (min = 1981, max = 2019)
    :param endyear: the last year of data to be  downloaded (min = 1981, max = 2019)
    :param latmin: the minimum latitude value of the region (float)
    :param latmax: the maximum latitude value of the region (float)
    :param lonmin: the minimum longitude value of the region (float)
    :param lonmax: the maximum longitude value of the region (float)
    :param regionname: name of the region (it could be any name the user wants) (string)
    :param t_resolution: the time resolution to be downloaded (daily or hourly)
    :param data_path:  the file path that contain the downloaded data (string)
    :param output_path:  the file path to store the extracted subset of the data (string)
    :return:
    """

    if t_resolution == 'daily':
        datapath = data_path 
    elif t_resolution == 'hourly':
        print('hourly', data_path)
        datapath = data_path
    else:
        raise ValueError("t_resolution is wrong please write 'daily' or 'hourly'")

    # set up the year array loop through each year to download the data
    years = np.arange(startyear,endyear+1)
    for y in range(0,len(years)):
        year=int(years[y])
        region_extract(datapath,year,latmin,latmax,lonmin,lonmax,regionname,t_resolution,output_path)
        print(year)


def wrapper_singlepoint(startyear,endyear,latval,lonval,regionname,t_resolution,data_path,output_path):
    """
    This is a wrapper function to run for downloading hPET and dPET data.
    All the arguments need to be given at the end of the script before running
    the script.

    :param startyear: the begging year to start the data download (min = 1981, max = 2019)
    :param endyear: the last year of data to be  downloaded (min = 1981, max = 2019)
    :param latmin: the minimum latitude value of the region (float)
    :param latmax: the maximum latitude value of the region (float)
    :param lonmin: the minimum longitude value of the region (float)
    :param lonmax: the maximum longitude value of the region (float)
    :param regionname: name of the region (it could be any name the user wants) (string)
    :param t_resolution: the time resolution to be downloaded (daily or hourly)
    :param data_path:  the file path that contain the downloaded data (string)
    :param output_path:  the file path to store the extracted subset of the data (string)
    :return:
    """

    if t_resolution == 'daily':
        datapath = data_path
    elif t_resolution == 'hourly':
        datapath = data_path
    else:
        raise ValueError("t_resolution is wrong please write 'daily' or 'hourly'")

    # set up the year array loop through each year to download the data
    years = np.arange(startyear,endyear+1)
    for y in range(0,len(years)):
        year=int(years[y])
        singlepoint_extract(datapath,year,latval,lonval,regionname,t_resolution,output_path)
        print(year)


def region_extract(datapath,year,latmin,latmax,lonmin,lonmax,regionname,t_resolution,output_path):
    """
    This function extract the data from the global hPET and dPET file and write a new
    netCDF file with a file name <year>_<t_resolution>_pet_<regionname>.nc in the output_path
    provided.

    :param datapath: the file path where the hPET data is stored (url)
    :param year: the year for which data is going to be downloaded (integer)
    :param latmin: the minimum latitude value (float)
    :param latmax: the maximum latitude value (float)
    :param lonmin: the minimum longitude value (float)
    :param lonmax: the maximum longitude value (float)
    :param regionname: name of the region (it could be any name the user wants) (string)
    :param t_resolution: the time resolution to be downloaded (daily or hourly)
    :param output_path:  the file path to store the downloaded data (string)
    :return: hPET or dPET data in a netCDF file
    """

    if t_resolution == 'daily':
        fname = '_daily_pet.nc'
        tunits='days since '+str(year)+'-01-01' # time unit for the new netcdf file
    elif t_resolution == 'hourly':
        fname = '_hourly_pet.nc'
        tunits='hours since '+str(year)+'-01-01 00:00:00'
    else:
        raise ValueError("t_resolution is wrong please write 'daily' or 'hourly'")

    pet_hr = Dataset(datapath + str(year) + fname)
    lats = pet_hr.variables['latitude'][:]
    lons = pet_hr.variables['longitude'][:]
    
    # extract the min and max index
    latminind, lonminind = nearest_point(latmin, lonmin, lats, lons)
    latmaxind, lonmaxind = nearest_point(latmax, lonmax, lats, lons)
 
    # read the data pet
    reg_data=pet_hr.variables['pet'][:, latmaxind:latminind, lonminind:lonmaxind]  
    # read the new latitude and longitude
    newlats=lats[latmaxind:latminind]
    newlons=lons[lonminind:lonmaxind]
    # get a filename and variable name (here it is called pet)
    filename=output_path+str(year)+'_'+t_resolution+'_pet_'+regionname+'.nc'
    varname='pet'
    # write the new data on a netcdf file
    nc_write(reg_data, newlats, newlons, varname, tunits, filename)
    
    # Extract the timezone values for the grid
    nc_offset=Dataset(datapath + 'timezone_offset.nc')
    offset=nc_offset.variables['offset'][latmaxind:latminind, lonminind:lonmaxind]
    # write the new data on a netcdf file
    filename = output_path+'timezone_offset_'+regionname+'.nc'
    tunits = 'days since 1981-01-01'
    nc_write(offset, newlats, newlons, 'offset', tunits, filename)
    
    return None


def singlepoint_extract(datapath,year,latval,lonval,regionname,t_resolution,output_path):
    """
    This function extract the data from the global hPET and dPET file and write a new
    netCDF file with a file name <year>_<t_resolution>_pet_<regionname>.nc in the output_path
    provided.

    :param datapath: the file path where the hPET data is stored (url)
    :param year: the year for which data is going to be downloaded (integer)
    :param latmin: the minimum latitude value (float)
    :param latmax: the maximum latitude value (float)
    :param lonmin: the minimum longitude value (float)
    :param lonmax: the maximum longitude value (float)
    :param regionname: name of the region (it could be any name the user wants) (string)
    :param t_resolution: the time resolution to be downloaded (daily or hourly)
    :param output_path:  the file path to store the downloaded data (string)
    :return: hPET or dPET data in a netCDF file
    """

    if t_resolution == 'daily':
        fname = '_daily_pet.nc'
        tunits='days since '+str(year)+'-01-01' # time unit for the new netcdf file
        
    elif t_resolution == 'hourly':
        fname = '_hourly_pet.nc'
        tunits='hours since '+str(year)+'-01-01 00:00:00'
        
    else:
        raise ValueError("t_resolution is wrong please write 'daily' or 'hourly'")

    pet_hr = Dataset(datapath + str(year) + fname)
    lats = pet_hr.variables['latitude'][:]
    lons = pet_hr.variables['longitude'][:]
    
    # extract the min and max index
    latind, lonind = nearest_point(latval, lonval, lats, lons)
 
    print(latind)
    print(lonind)
    # read the data pet
    point_data=pet_hr.variables['pet'][:, latind, lonind]  

    # Extract the timezone values for the grid
    nc_offset=Dataset(datapath + 'timezone_offset.nc')
    offset=nc_offset.variables['offset'][latind, lonind]
    if t_resolution == 'daily':
        filename='dPET_'+str(latval)+'_'+str(lonval)+'_'+str(year)+'.txt'
    elif t_resolution == 'hourly':
        filename='hPET_'+str(latval)+'_'+str(lonval)+'_'+str(offset)+'_'+str(year)+'.txt'
    else:
        raise ValueError("t_resolution is wrong please write 'daily' or 'hourly'")
    # save the data in a text file
    np.savetxt(output_path+filename,point_data,fmt='%0.5f')
    
    return None   


def nearest_point(lat_var, lon_var, lats, lons):
    """
    This function identify the nearest grid location index for a specific lat-lon
    point.
    :param lat_var: the latitude
    :param lon_var: the longitude
    :param lats: all available latitude locations in the data
    :param lons: all available longitude locations in the data
    :return: the lat_index and lon_index
    """
    # this part is to handle if lons are givn 0-360 or -180-180
    if any(lons > 180.0) and (lon_var < 0.0):
        lon_var = lon_var + 360.0
    else:
        lon_var = lon_var
        
    lat = lats
    lon = lons

    if lat.ndim == 2:
        lat = lat[:, 0]
    else:
        pass
    if lon.ndim == 2:
        lon = lon[0, :]
    else:
        pass

    index_a = np.where(lat >= lat_var)[0][-1]
    index_b = np.where(lat <= lat_var)[0][-1]

    if abs(lat[index_a] - lat_var) >= abs(lat[index_b] - lat_var):
        index_lat = index_b
    else:
        index_lat = index_a

    index_a = np.where(lon >= lon_var)[0][0]
    index_b = np.where(lon <= lon_var)[0][0]
    if abs(lon[index_a] - lon_var) >= abs(lon[index_b] - lon_var):
        index_lon = index_b
    else:
        index_lon = index_a

    return index_lat, index_lon


def nc_write(data, lat, lon, varname, tunits, filename):
    """
    this function write the PET on a netCDF file.

    :param: data: data to be written (time,lat,lon)
    :param: lat: latitude
    :param: lon: longitude
    :param: varname: name of the variable to be written (e.g. 'pet')
    :param: tunits: time units for the data (e.g. 'days since 1981-01-01')
    :param:filename: the file name to write the values with .nc extension

    :return:  produce a netCDF file in the same directory.
    """
    
    ds = Dataset(filename, mode='w', format='NETCDF4_CLASSIC')

    time = ds.createDimension('time', None)
    latitude = ds.createDimension('latitude', len(lat))
    longitude = ds.createDimension('longitude', len(lon))
   
    time = ds.createVariable('time', np.float32, ('time',))
    latitude = ds.createVariable('latitude', np.float32, ('latitude',))
    longitude = ds.createVariable('longitude', np.float32, ('longitude',))

    # check if the data is 2d or 3d
    if len(data.shape) == 3: # 3D array
        pet_val = ds.createVariable(varname, 'f4', ('time','latitude','longitude'), zlib=True)
        time.units = tunits  
        time.calendar = 'proleptic_gregorian'
        time[:] = np.arange(data.shape[0])
        latitude[:] = lat
        longitude [:] = lon
        pet_val[:,:,:] = data
    # this is only to write the time offsets
    elif len(data.shape) == 2: # 2D array
        pet_val = ds.createVariable(varname, 'i', ('latitude','longitude'), zlib=True)
        latitude[:] = lat
        longitude [:] = lon
        pet_val[:,:] = data
    else:
        raise ValueError('the function can only write a 2D or 3D array data!')

    ds.close()
    
    return None    

##********************************************************************************##
if __name__ == '__main__':
    start = dt.datetime.now()
    main()
    end = dt.datetime.now()
    print(end-start)

# HOSTRADA and mPET
 - both are provided publicly by DWD, so similar functions work for downloading
 - start and end hardcoded on function get_links(), should be changed based on HOSTRADA and mPET

## downloading #

In [ ]:
import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from multiprocessing import Pool
from tqdm import tqdm
import glob, os
import numpy as np
from netCDF4 import Dataset
import datetime as dt
import pandas as pd
import xarray as xr 
def get_links(url, extension):
    '''
    Gets downloadable links for datasets within years
    '''
    print("Getting links")
    start = 5 #1 for HOSTRADA, 5 for monthly pet
    end = -2  #-1 for HOSTRADA, -2 for monthly pet 
    
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    file_urls = []
    filenames = []

    for link in soup.find_all("a", href=True)[start: end]: #hardcoded as dwd link contains parent dir as first link
        href = link["href"]
        print(href)
        fname = href.split('_')[5][:4]  #this naming structure is hardcoded
        if (href.lower().endswith(extension) and any(year in fname for year in years)):
            full_url = urljoin(url, href)
            file_urls.append(full_url)
            filenames.append(href)
#         break

    return file_urls, filenames

def download_file(args):
    '''
    Puts a request to download specific file 
    url, filename in args. taken as tuple to ease multiprocessing.
    '''
    url, filename = args
    save_path = os.path.join(save_dir, filename)

    try:
        with requests.get(url, stream=True, timeout=30) as r:
            r.raise_for_status()

            total = int(r.headers.get("content-length", 0))
            progress = tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                desc=filename,
                leave=False
            )

            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=32768): #32kb chunks
                    if chunk:
                        f.write(chunk)
                        progress.update(len(chunk))

            progress.close()
        return f"Saved {filename}"

    except Exception as e:
        return f"ERROR {filename}: {e}"

## hostrada

In [ ]:
url = r'https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/air_temperature_mean/'
save_dir = r"D:\hostrada"
max_workers = 6
years = [str(y) for y in range(2003, 2023)]
extension = (".nc")
if __name__ == "__main__":
    file_urls, filenames = get_links()
    tasks = list(zip(file_urls, filenames))
    with Pool(max_workers) as pool:
        results = pool.map(download_file, tasks)

## mpet 

In [ ]:
url = r'https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/evapo_p/'
save_dir = r"D:\hostrada_all_dataset\mointhly_asci"
max_workers = 6
years = [str(y) for y in range(2001, 2023)]
extension = (".asc.gz")
if __name__ == "__main__":
    file_urls, filenames = get_links()
    tasks = list(zip(file_urls, filenames))
    with Pool(max_workers) as pool:
        results = pool.map(download_file, tasks)

#### converting mpet from ascii to netcdf, geocoding, and getting annual netcdf

In [ ]:
def ascii2nc(file_path, dst_dir):
    '''
    Convers monthly pet ascii file to nc.
    based on https://code.mpimet.mpg.de/boards/1/topics/3631
    '''
    #from ascii file 
    ncols =           654
    nrows   =       866
    xllcorner =     3280414
    yllcorner =    5237501
    cellsize  =    1000
    xllcgrid = xllcorner + cellsize/2 #grid centre
    yllcgrid = yllcorner + cellsize/2
    
    infile_name = os.path.basename(file_path).split('.')[0]
    year = int(infile_name.split('_')[-1][:4])
    month = int(infile_name.split('_')[-1][4:6])
    day = 1
    print(year, month, day, end = ': ')
    outfile_path = os.path.join(dst_dir, f'{infile_name}.nc')
    if os.path.exists(outfile_path):
        os.remove(outfile_path) #avoid permission error 
    with Dataset(outfile_path, 'w', format='NETCDF4') as root:
        root.description = 'PET in mm'
        root.createDimension('y',nrows) #y is row, 
        root.createDimension('x',ncols) #x is col,
        root.createDimension('time',1)
        x = root.createVariable('x', 'f4', ('x',))
        y = root.createVariable('y', 'f4', ('y',))
        time = root.createVariable('time', 'str', ('time',))
        date = dt.date(year, month, day)
        print(date)
        time[0] = date.strftime('%Y-%m-%d')
        pet = root.createVariable('pet', 'f4', ('y','x'),fill_value=-9999/10) #AP: a+(n-1)*d but i indexed from 0 so i = n-1
        lats_list = [yllcgrid + (i)*(cellsize) for i in range(nrows)]
        lats_list.reverse() #y values populated from top to bottom, so reversed
        lats = np.array(lats_list)
        # len(lats)
        lons = np.array([xllcgrid + (i)*(cellsize) for i in range(ncols)]) 
        # len(lons)
        # len(lats[0:700])
        y[:] = lats
        x[:] = lons
        f = open(file_path)
        a = np.loadtxt(f, skiprows=6)
        pet[:,:] = a[:,:]/10 #scaled to mm from cm. based on documentation. 
def convert_allascii(src_dir = r'D:\hostrada_all_dataset\1_ascii',
                     dst_dir = r'D:\hostrada_all_dataset\2_nc'):
    files = glob.glob(os.path.join(src_dir, '*.asc'))
    for file in files:
        ascii2nc(file, dst_dir)
        
def get_annual_nc(src_dir = r'D:\hostrada_all_dataset\2_nc',
                  dst_dir = r'D:\hostrada_all_dataset\2_nc\annual',
                  years = range(2001,2023),
                  file_name_regex = 'grids_germany_monthly_evapo_p_'):
    '''
    Combines all months of a year into yearly dataset
    '''
    for year in years:
        open_files = glob.glob(os.path.join(src_dir, 
                                            f'{file_name_regex}{year}*.nc'))
        save_file = os.path.join(dst_dir, f'{year}.nc')
        print(year, '\n',[os.path.basename(_) for _ in open_files])
        ds = xr.open_mfdataset(open_files, concat_dim="time", 
                               combine = 'nested', parallel = False)
        ds.to_netcdf(save_file)

# comparing the datasets

## pet dataset plots
- these functions create comparative plots for all pet datasets
    - cumulative pet for year plotted 
    - cumulative pet for each month for all years are plotted
    - average pet for each hour in each month are averaged and plotted 
    - average pet for each hour and its range is plotted 

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt 
def prepare_df(file_path):
    df = pd.read_csv(file_path, delimiter = '\t', header = None)
    if not file_path.__repr__().lower() in ['metref', 'mPet']:
        df.columns = ['year', 'month', 'day', 'hour', 'pev']
        return df
    else:
        df.columns = ['year', 'month', 'day', 'hour', 'pev']
        df.drop(columns = 'hour', inplace = True)
        return df

def plot_annualsum(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    # plt.style.use('bmh')
    fig, ax = plt.subplots(figsize=(15, 7))

    for df, label in df_label:
        grouped = df.groupby('year')['pev'].sum()
        ax.plot(grouped.index, grouped.values, 
                label=label, 
                linewidth=2.5, 
                marker='o', 
                markersize=6, 
                alpha=0.8)
    
    ax.grid(visible=True, which='major', axis='both', linestyle='--', alpha=0.6, color='gray')
    
    ax.text(0.02, 0.95, '*METREF starts from 2004-01-20', 
            transform=ax.transAxes, 
            fontsize=14, 
            fontweight='bold', 
            color='darkred',
            verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
    
    ax.set_xlabel('Year', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_ylabel('PET [mm]', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_title(f'Cumulative PET in {EZG_ID}', fontsize=16, fontweight='bold', pad=20)
    
    ax.set_xticks(np.arange(2001, 2023))
    plt.xticks(rotation=45)
    
    ax.legend(title="Datasets", bbox_to_anchor=(1, 1), loc='upper left', frameon=True)
    
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()
    
def plot_monthly_sum_all_yr(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    # plt.style.use('bmh')
    fig, ax = plt.subplots(figsize=(15, 7))

    for df, label in df_label:
        print(label)
        grouped = df.groupby(['year', 'month', 'day'])['pev'].sum().reset_index().groupby('month')['pev'].sum()
        # grouped = df.groupby('month')['pev'].sum()
        ax.plot(grouped.index, grouped.values, 
                label=label, 
                linewidth=2.5, 
                marker='o', 
                markersize=6, 
                alpha=0.8)
    
    ax.grid(visible=True, which='major', axis='both', linestyle='--', alpha=0.6, color='gray')
    
    # ax.text(0.02, 0.95, '*METREF starts from 2004', 
    #         transform=ax.transAxes, 
    #         fontsize=14, 
    #         fontweight='bold', 
    #         color='darkred',
    #         verticalalignment='top',
    #         bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
    
    ax.set_xlabel('Month', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_ylabel('PET [mm]', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_title(f'Seasonal Pattern of PET in {EZG_ID}', fontsize=16, fontweight='bold', pad=20)
    
    ax.set_xticks(np.arange(1, 13))
    plt.xticks(rotation=45)
    
    ax.legend(title="Datasets", bbox_to_anchor=(0.9, 1), loc='upper left', frameon=True)
    
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()

def plot_hourly_mean_all_yr(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(figsize=(15, 7))
    for df, label in df_label:
        if label not in  ['metref', 'mpet']:
            grouped = df.groupby(['hour'])['pev'].mean() #mean()
            grouped_min = df.groupby(['hour'])['pev'].min()
            grouped_max = df.groupby(['hour'])['pev'].max()
            
            axes.plot(grouped.index, grouped.values,
                    label=label,
                    linewidth=2.5,
                    marker='o',
                    markersize=6,
                    alpha=0.8)
            axes.fill_between(grouped_min.index, grouped_min, grouped_max,
                              alpha=0.15,
                              label='range')
                
    # handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(title="Datasets", bbox_to_anchor=(0.83, 0.85), loc='center left', frameon=True)
    # fig.legend(title="Datasets", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    axes.set_title(f'Hourly Trend \n Catchment: {EZG_ID}', fontsize=16, fontweight='bold')
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()

def plot_hourly_mean_monthly_all_yr(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(3, 4, figsize=(15, 10), sharex=True, sharey=True)
    for df, label in df_label:
        if label not in  ['metref', 'mpet']:
            # grouped = df.groupby(['year', 'month', 'day'])['pev'].sum().reset_index().groupby('month')['pev'].mean()
            grouped = df.groupby(['month', 'hour'])['pev'].mean()#sum() #mean()
            grouped_min = df.groupby(['month', 'hour'])['pev'].min()
            grouped_max = df.groupby(['month', 'hour'])['pev'].max()
            
            axes = axes.flatten()
            month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
            
            for i, month in enumerate(range(1, 13)):
                m_mean = grouped.loc[month]
                m_min = grouped_min.loc[month]
                m_max = grouped_max.loc[month]
                
                line, = axes[i].plot(m_mean.index, m_mean.values, lw=1.5, label=label,marker='o', markersize = 2)
                # shade = axes[i].fill_between(m_mean.index, m_min, m_max, color='blue', alpha=0.2, label='Range')
                axes[i].set_title(month_names[i])
                axes[i].set_xticks(range(0, 24, 4))
                if i >= 8: axes[i].set_xlabel('Hour')
                if i % 4 == 0: axes[i].set_ylabel('PET (mm)')
    # fig.legend([line, shade], ['Mean', 'Min-Max Range'], loc='upper right', bbox_to_anchor=(1, 1.001))
                
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Datasets", bbox_to_anchor=(0.83, 0.95), loc='center left', frameon=True)
    # fig.legend(title="Datasets", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    fig.suptitle(f'Hourly Trend within Months \n Catchment: {EZG_ID}', fontsize=16, fontweight='bold')
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()




def summarize_etos(shapefile_path = r".\New Folder\catchments_all.shp",
                   era5land_dir = r'.\pet\ERA5_Land\catchment_pev\data\hbv' ,
                   metref_dir = r'.\pet\ls_saf\eto\data\hbv',
                   cerra_dir = r'.\pet\cerra\catchment_results\eto',
                   hpet_dir = r'.\pet\hpet_actual\catchment_outputs\eto',
                  mpet_dir = r'.\pet\mPet_1km\pet',
                  era5_dir = r'.\pet\ERA5\outputs\eto'):
    '''
    this is the only function where the variables must be declared. 
    shapefile_path: path to shapefile
    all_dirs are the directories where .eto is stored. eto for different dataset should be stored in different dir.
    '''
    dirs = [era5_dir, era5land_dir, metref_dir, cerra_dir, hpet_dir, mpet_dir]
    product_names = ['era5', 'era5land', 'metref', 'cerra', 'hpet', 'mpet']
    catchments = gpd.read_file(shapefile_path)
    for num in np.arange(0,catchments.shape[0],1):
        EZG_ID = catchments.Pegel_ID.iloc[num]
        catchment = catchments[catchments.Pegel_ID == EZG_ID]
        eto_file_pattern = [os.path.join(dir_path, f'*{int(EZG_ID)}*.eto')
                           for dir_path in dirs]
        files = [glob.glob(file_pattern)
                 for file_pattern in eto_file_pattern]
        # print(eto_file_pattern)
        # print(files)
        if np.all(np.array([len(_) for _ in files])): #make sure all files exist
            era5, era5l, metref, cerra, hpet, mpet = [prepare_df(_[0]) for _ in files]
            plot_annualsum(zip([era5, era5l, metref, cerra, hpet, mpet],
                              ['era5', 'era5land', 'metref', 'cerra', 'hpet', 'mpet']),
                           int(EZG_ID),
                           save_dir = r'.\pet\summary\annual_sums')
            
            plot_monthly_sum_all_yr(zip([era5, era5l, metref, cerra, hpet, mpet],
                                        ['era5', 'era5land', 'metref', 'cerra', 'hpet', 'mpet']),
                                    int(EZG_ID),
                                    save_dir = r'.\pet\summary\monthly_sums')
            
            plot_hourly_mean_monthly_all_yr(zip([era5, era5l, metref, cerra, hpet, mpet],
                                                ['era5', 'era5land', 'metref', 'cerra', 'hpet', 'mpet']),
                                            int(EZG_ID),
                                            save_dir = r'.\pet\summary\hourly_means')

            
            plot_hourly_mean_all_yr(zip([era5, era5l, metref, cerra, hpet, mpet],
                                        ['era5', 'era5land', 'metref', 'cerra', 'hpet', 'mpet']),
                                    int(EZG_ID),
                                    save_dir = r'.\pet\summary\hourly_trend')
        else:
            print(f'{EZG_ID} data remainig')
        # break
    # return file_label
        # print(era5land, metref, cerra, hpet) 
        # if num > 1:
        #     break
        
summarize_etos()

## t2m dataset plots
- these functions create comparative plots for all t2m datasets
    - average t2m for year plotted 
    - averate t2m for each month for all years are plotted
    - average t2m for each hour in each month are averaged and plotted 
    - average t2m for each hour and its range is plotted 

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import time 
def plot_annualmean(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    # plt.style.use('bmh')
    fig, ax = plt.subplots(figsize=(15, 7))

    for df, label in df_label:
        grouped = df.groupby('year')['t2m'].mean()
        ax.plot(grouped.index, grouped.values, 
                label=label, 
                linewidth=2.5, 
                marker='o', 
                markersize=6, 
                alpha=0.8)
    
    ax.grid(visible=True, which='major', axis='both', linestyle='--', alpha=0.6, color='gray')
    
    ax.set_xlabel('Year', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_ylabel('t2m [°C]', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_title(f'Mean 2m Temperature in {EZG_ID}', fontsize=16, fontweight='bold', pad=20)
    
    ax.set_xticks(np.arange(2001, 2023))
    plt.xticks(rotation=45)
    
    ax.legend(title="Datasets", bbox_to_anchor=(1, 1), loc='upper left', frameon=True)
    
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()
    
def plot_monthly_mean_all_yr(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    # plt.style.use('bmh')
    fig, ax = plt.subplots(figsize=(15, 7))

    for df, label in df_label:
        grouped = df.groupby(['year', 'month', 'day'])['t2m'].mean().reset_index().groupby('month')['t2m'].mean()
        # grouped = df.groupby('month')['pev'].sum()
        ax.plot(grouped.index, grouped.values, 
                label=label, 
                linewidth=2.5, 
                marker='o', 
                markersize=6, 
                alpha=0.8)
    
    ax.grid(visible=True, which='major', axis='both', linestyle='--', alpha=0.6, color='gray')
    
    # ax.text(0.02, 0.95, '*METREF starts from 2004', 
    #         transform=ax.transAxes, 
    #         fontsize=14, 
    #         fontweight='bold', 
    #         color='darkred',
    #         verticalalignment='top',
    #         bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
    
    ax.set_xlabel('Month', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_ylabel('t2m [°C]', fontsize=15, fontweight='bold', labelpad=10)
    ax.set_title(f'Seasonal Pattern of PET in {EZG_ID}', fontsize=16, fontweight='bold', pad=20)
    
    ax.set_xticks(np.arange(1, 13))
    plt.xticks(rotation=45)
    
    ax.legend(title="Datasets", bbox_to_anchor=(0.9, 1), loc='upper left', frameon=True)
    
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()

def plot_hourly_mean_all_yr(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(figsize=(15, 7))
    for df, label in df_label:
        if label != 'metref':
            grouped = df.groupby(['hour'])['t2m'].mean() #mean()
            grouped_min = df.groupby(['hour'])['t2m'].min()
            grouped_max = df.groupby(['hour'])['t2m'].max()
            
            axes.plot(grouped.index, grouped.values,
                    label=label,
                    linewidth=2.5,
                    marker='o',
                    markersize=6,
                    alpha=0.8)
            axes.fill_between(grouped_min.index, grouped_min, grouped_max,
                              alpha=0.15,
                              label='range')
                
    # handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(title="Datasets", bbox_to_anchor=(0.83, 0.85), loc='center left', frameon=True)
    # fig.legend(title="Datasets", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    axes.set_title(f'Hourly Trend \n Catchment: {EZG_ID}', fontsize=16, fontweight='bold')
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()

def plot_hourly_mean_monthly_all_yr(df_label,EZG_ID,save_dir):
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(3, 4, figsize=(15, 10), sharex=True, sharey=True)
    for df, label in df_label:
        if label != 'metref':
            # grouped = df.groupby(['year', 'month', 'day'])['pev'].sum().reset_index().groupby('month')['pev'].mean()
            grouped = df.groupby(['month', 'hour'])['t2m'].mean()#sum() #mean()
            grouped_min = df.groupby(['month', 'hour'])['t2m'].min()
            grouped_max = df.groupby(['month', 'hour'])['t2m'].max()
            
            axes = axes.flatten()
            month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
            
            for i, month in enumerate(range(1, 13)):
                m_mean = grouped.loc[month]
                m_min = grouped_min.loc[month]
                m_max = grouped_max.loc[month]
                
                line, = axes[i].plot(m_mean.index, m_mean.values, lw=1.5, label=label,marker='o', markersize = 2)
                # shade = axes[i].fill_between(m_mean.index, m_min, m_max, color='blue', alpha=0.2, label='Range')
                axes[i].set_title(month_names[i])
                axes[i].set_xticks(range(0, 24, 4))
                if i >= 8: axes[i].set_xlabel('Hour')
                if i % 4 == 0: axes[i].set_ylabel('t2m [°C]')
    # fig.legend([line, shade], ['Mean', 'Min-Max Range'], loc='upper right', bbox_to_anchor=(1, 1.001))
                
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Datasets", bbox_to_anchor=(0.83, 0.95), loc='center left', frameon=True)
    # fig.legend(title="Datasets", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
    fig.suptitle(f'Hourly Trend within Months \n Catchment: {EZG_ID}', fontsize=16, fontweight='bold')
    # plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{EZG_ID}.png'), bbox_inches='tight')
    plt.show()


def prepare_df(file_path):
    df = pd.read_csv(file_path, delimiter = '\t', header = None)
    if not 'metref' in file_path.__repr__().lower():
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        return df
    else:
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        df.drop(columns = 'hour', inplace = True)
        return df

def summarize_t2ms(shapefile_path = r".\New Folder\catchments_all.shp",
                   era5_dir = r'.\pet\ERA5\tav' ,
                   hostrada_dir = r'.\pet\HOSTRADA\tav',
                   cerra_dir = r'.\pet\CERRA\tav',
                   era5land_dir = r'.\pet\ERA5_Land\tav'):
    '''
    this is the only function where the variables must be declared. 
    shapefile_path: path to shapefile
    all_dirs are the directories where .eto is stored. eto for different dataset should be stored in different dir.
    '''
    dirs = [era5_dir, era5land_dir, cerra_dir, hostrada_dir]
    product_names = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA']
    catchments = gpd.read_file(shapefile_path)
    for num in np.arange(0,catchments.shape[0],1):
        EZG_ID = catchments.Pegel_ID.iloc[num]
        catchment = catchments[catchments.Pegel_ID == EZG_ID]
        eto_file_pattern = [os.path.join(dir_path, f'*{int(EZG_ID)}*.tav')
                           for dir_path in dirs]
        files = [glob.glob(file_pattern)
                 for file_pattern in eto_file_pattern]
#         print(files)
        if np.all(np.array([len(_) for _ in files])):
            if not os.path.exists(os.path.join(r'.\pet\summary\annual_mean', f'{int(EZG_ID)}.png')): #make sure all files exist
                era5l, metref, cerra, hostrada = [prepare_df(_[0]) for _ in files]
                plot_annualmean(zip([era5l, metref, cerra, hostrada],
                                   product_names), 
                               int(EZG_ID),
                              save_dir = r'.\pet\summary\annual_mean')

                plot_monthly_mean_all_yr(zip([era5l, metref, cerra, hostrada],
                                            product_names), 
                                        int(EZG_ID),
                                       save_dir = r'.\pet\summary\monthly_mean')

                plot_hourly_mean_monthly_all_yr(zip([era5l, metref, cerra, hostrada],
                                                    product_names), 
                                                int(EZG_ID),
                                               save_dir = r'.\pet\summary\hourly_mean_monthly')

                plot_hourly_mean_all_yr(zip([era5l, metref, cerra, hostrada],
                                                    product_names), 
                                                int(EZG_ID),
                                               save_dir = r'.\pet\summary\hourly_mean_all_year')
            
        else:
            print(f'{EZG_ID} data remainig')
#         break
        # print(era5land, metref, cerra, hpet) 
#         if num > 4:
#             break
# time.sleep(60*60*2)
summarize_t2ms()

#### t2m min max plot
- idea was checking t2m of models at hottest (July) and coldest (Jan) month
- too many plots to create anything conclusive, but can be used to check performance over certain periods of interest 

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import time
def prepare_df(file_path):
    df = pd.read_csv(file_path, delimiter = '\t', header = None)
    if not 'metref' in file_path.__repr__().lower():
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        return df
    else:
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        df.drop(columns = 'hour', inplace = True)
        return df
def min_t2ms(shapefile_path = r".\New Folder\catchments_all.shp",
                   era5_dir = r'.\pet\ERA5\tav' ,
                   hostrada_dir = r'.\pet\HOSTRADA\tav',
                   cerra_dir = r'.\pet\CERRA\tav',
                   era5land_dir = r'.\pet\ERA5_Land\tav'):
    '''
    this is the only function where the variables must be declared. 
    shapefile_path: path to shapefile
    all_dirs are the directories where .eto is stored. eto for different dataset should be stored in different dir.
    '''
    dirs = [era5_dir, era5land_dir, cerra_dir, hostrada_dir]
    product_names = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA']
    catchments = gpd.read_file(shapefile_path)
    for num in np.arange(0,catchments.shape[0],1):
        EZG_ID = catchments.Pegel_ID.iloc[num]
        if True:#int(EZG_ID) == 4822106:
            catchment = catchments[catchments.Pegel_ID == EZG_ID]
            eto_file_pattern = [os.path.join(dir_path, f'*{int(EZG_ID)}*.tav')
                               for dir_path in dirs]
            files = [glob.glob(file_pattern)
                     for file_pattern in eto_file_pattern]
            if np.all(np.array([len(_) for _ in files])): #make sure all files exist
    #             era5l, metref, cerra, hpet = [prepare_df(_[0]) for _ in files]
                print('Reading files')        
                era5l, metref, cerra, hostrada = [prepare_df(_[0]) for _ in files]
                iter_item = zip([era5l, metref, cerra, hostrada], 
                                product_names)
                plt.style.use("default")
                fig, ax = plt.subplots(figsize=(17, 7))#plt.plot(figsize = (17,5))

                for df, label_name in iter_item:
                    x = df.t2m[:24*31] ##hardcoded for Jan 2001
                    ax.plot(
                        x,
                        label=label_name,
                        linewidth=1.4,
    #                     alpha=0.9
                    )
                ax.set_title("Hourly Temperature Over January 2001", fontsize=18, pad=15)
                
                
                dt = pd.to_datetime({
                    "year":  df.year[:24*31],
                    "month": df.month[:24*31],
                    "day":   df.day[:24*31],
                    "hour":  df.hour[:24*31]})
                tick_positions = np.arange(0, len(dt), 24*7)
                tick_labels = dt.iloc[tick_positions].dt.strftime("%d %b")

                ax.set_xticks(tick_positions)
                ax.set_xticklabels(tick_labels, rotation=45)
                ax.set_xlabel("Date", fontsize=14)
                
                ax.set_xlabel("Hour of January", fontsize=14)
                ax.set_ylabel("Temperature (°C)", fontsize=14)
#                 ax.set_ylim(-15, 15)
                ax.grid(True, linestyle="--", alpha=0.4)
                ax.legend(title="Dataset", fontsize=12, title_fontsize=12)
                fig.tight_layout()
                fig.savefig(os.path.join(r'.\pet\summary\Jan_2001', f'{int(EZG_ID)}.png'))
                plt.show()

                ##for july the hottest month
                iter_item = zip([era5l, metref, cerra, hostrada], 
                                product_names)
                plt.style.use("default")
                fig, ax = plt.subplots(figsize=(17, 7))#plt.plot(figsize = (17,5))
                nhrs_julystart = (31+28+31+30+31+30)*24
                nhrs_julyend = (31+28+31+30+31+30+31)*24
                for df, label_name in iter_item:
                    x = df.t2m[nhrs_julystart:nhrs_julyend] ##hardcoded for Jan 2001
                    ax.plot(
                        x,
                        label=label_name,
                        linewidth=1.4,
    #                     alpha=0.9
                    )
                ax.set_title("Hourly Temperature Over July 2001", fontsize=18, pad=15)
                
                dt = pd.to_datetime({
                    "year":  df.year[nhrs_julystart:nhrs_julyend],
                    "month": df.month[nhrs_julystart:nhrs_julyend],
                    "day":   df.day[nhrs_julystart:nhrs_julyend],
                    "hour":  df.hour[nhrs_julystart:nhrs_julyend]})
                tick_positions_df = np.arange(nhrs_julystart, nhrs_julyend, 24*7)
                tick_positions = np.arange(0, len(dt), 24*7)
                print(tick_positions)
                tick_labels = dt.iloc[tick_positions].dt.strftime("%d %b")

                ax.set_xticks(tick_positions_df)
                ax.set_xticklabels(tick_labels, rotation=45)
                ax.set_xlabel("Date", fontsize=14)
                ax.set_ylabel("Temperature (°C)", fontsize=14)
#                 ax.set_ylim(0, 38)
                ax.grid(True, linestyle="--", alpha=0.4)
                ax.legend(title="Dataset", fontsize=12, title_fontsize=12)
                fig.tight_layout()
                fig.savefig(os.path.join(r'.\pet\summary\Jul_2001', f'{int(EZG_ID)}.png'))
                plt.show()
#                 break
#                 break
    return [era5l, metref, cerra, hostrada]        
dfs = min_t2ms()

## metric plots
- creates metric matrix for each catchment for all pet datasets
- metrices calculated by taking all data at once, summer, winter, spring and autumn months separately 

#### PET
- PBIAS, MAE, correlation with respect to one another, violin plot for distribution of the datasets
    - cumulative monthly (ERA5, ERA5Land, hPET, CERRA, METREF, mPET), 
    - cumulative daily (ERA5, ERA5Land, hPET, CERRA, METREF), 
    - and hourly pet (ERA5, ERA5Land, hPET, CERRA)
- correlation
    - if all datasets normal Pearson
        - else spearman

In [ ]:
from scipy.stats import normaltest
from scipy.stats import spearmanr
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import time
import seaborn as sns
import math
def prepare_df(file_path):
    '''
    pet col named as t2m, but doesn't matter, param hardcoded as t2m in other sections also
    '''
    df = pd.read_csv(file_path, delimiter = '\t', header = None)
    if not 'metref' in file_path.__repr__().lower():
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        return df
    else:
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        df.drop(columns = 'hour', inplace = True)
        return df
def get_pbias(x,y):
    numenor = np.sum(y-x)
#     denumenor =  np.sum(np.abs(x)) #np.sum(x)# #done so that if denumenor sum is -ve, it doesn't cancel out underestimation in numenor.
    denumenor = np.sum(x)
    #     print(numenor ,denumenor)
    return (numenor/denumenor)*100
def get_seasonalt2ms(dfs_o, months = list(range(1,13))):
    dfs = [df.copy(deep = True) for df in dfs_o]
    df_seasons = [df[df['month'].isin(months)] for df in dfs]
    return df_seasons
def get_corrmat(df_seasons): 
    t2ms = np.array([np.array(df.t2m) for df in df_seasons])
    ntest_pvalues = np.array([normaltest(t2m).pvalue for t2m in t2ms]) #scipy default based on kurtosis and skewness 
    if np.all(ntest_pvalues > 0.05):
#         print('All data normal', ntest_pvalues)
        test_type = 'Pearson'
        corr_matrix = np.round(np.corrcoef(t2ms),4)
    else:
#         print('All data non normal', ntest_pvalues)
        test_type = "Spearman"
#         corr_matrix = np.round(spearmanr(t2ms),4)
        corr_matrix = np.round(np.array([[spearmanr(df_ref.t2m, df_mod.t2m).statistic for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    return corr_matrix, test_type
    
def get_stats(df_seasons):
    results = dict()
    mae_matrix = np.round(np.array([[np.mean(np.abs(df_ref.t2m - df_mod.t2m)) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    pbias_matrix = np.round(np.array([[get_pbias(df_ref.t2m, df_mod.t2m) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 2)
    corr_matrix, test_type = get_corrmat(df_seasons)
    results['MAE'] = mae_matrix
    results['PBIAS'] = pbias_matrix
    results['correlation'] = corr_matrix
    results['corr_test'] = test_type
#     results['dfs'] = df_seasons
    return results


def plot_metrices(results_dict, EZG_ID,
                 labels = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA'],
                 metrics_order = ["MAE", "PBIAS", "correlation", "dfs"],
                 save_dir = r'D:\temperature_results\metrices'):
    #setting dicts for the plots
    correlation_type = results_dict['all seasons']['corr_test']
    metric_ranges = dict()
    metric_cmaps = {"MAE": 'cividis',#"mako",
                    "PBIAS": "coolwarm", #blue for underestimation, red for over 
                    "correlation": "RdYlGn"} #yl at 0, rd -1, gn +1 
    for metric in metrics_order:
        if metric == 'correlation':
            metric_ranges[metric] = (-1, 1)
        elif metric == 'PBIAS':
            metric_ranges[metric] = (-100, 100)
        elif metric == 'MAE':
            mats = [results_dict[d][metric] for d in list(results_dict.keys())]
            all_vals = np.concatenate([m.flatten() for m in mats])
            metric_ranges[metric] = (all_vals.min()*0.9, all_vals.max()*1.1) #10% addition 
        elif metric == 'dfs':
            mats = np.array([df.t2m.values for df in results_dict['all seasons'][metric]])
            metric_ranges[metric] = (math.floor(mats.min()*0.9), math.ceil(mats.max()*1.1))
    
    #setting up plot 
    nrows=len(metrics_order)
    ncols=len(list(results_dict.keys()))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(22, 20))
    for col, dtype in enumerate(list(results_dict.keys())):
        axes[0, col].set_title(dtype, fontsize=14, pad=30) #name of dtype in each column 

    for col, (dtype, dtype_dict) in enumerate(results_dict.items()):
        for row, metric in enumerate(metrics_order):
            ax = axes[row, col]
            if metric != 'dfs':
                mat = dtype_dict[metric]
                cmap = plt.get_cmap(metric_cmaps[metric]).copy()
                cmap.set_bad(color="lightgrey")
                vmin, vmax = metric_ranges[metric]
                if metric.lower() in ["mae", "correlation"]:
                    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
                    norm = None
                else:
                    mask = None
                    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
                    mask = np.eye(mat.shape[0], dtype=bool)
                mat_ma = np.ma.array(mat, mask=mask) #masked array
                if metric.lower() in ["mae", "pbias"]:
                    fmt=".2f"
                else:
                    fmt=".2f"
                hm = sns.heatmap(mat_ma, 
                                 annot=True, 
                                 fmt=fmt, 
                                 cmap=cmap, 
                                 ax=ax, 
                                 vmin=vmin, 
                                 vmax=vmax,
                                 norm = norm,
                                 cbar=False, 
                                 square=True, 
                                 mask=mask)
            
                if col == ncols - 1:
                    cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])
                    plt.colorbar(hm.collections[0], cax=cax)
                ax.set_xticks([])
#                 if row == nrows - 2:
#                     ax.set_xticks(np.arange(len(labels))+0.5)
#                     ax.set_xticklabels(labels, rotation=45, ha="right")
                ax.set_yticks([])
                if (col == 0) & (row != nrows-1):
                    ax.set_yticks(np.arange(len(labels))+0.5)
                    ax.set_yticklabels(labels, rotation=0)
                if row == 0:#nrows - 2:
                    ax.xaxis.tick_top()
                    ax.xaxis.set_label_position("top")
                    ax.set_xticks(np.arange(len(labels))+0.5)
                    ax.set_xticklabels(labels, rotation=90, ha="left")
                    ax.tick_params(axis="x", bottom=False)
            else:
                dfs = dtype_dict[metric]
                ax.yaxis.grid(True)
                ax.violinplot([df.t2m.values for df in dfs],showmeans=True)#, tick_labels=labels, patch_artist=True)
                ax.set_xticks([y+1 for y in range(len(labels))])
                ax.set_xticklabels(labels, rotation=90, ha="left")
#                 ax.set_ylabel('T [°C]')
#                 ax.set_xticks([])
#                 if col == 0:            
                vals = np.array([df.t2m.values for df in dfs])
                yticks = np.linspace(math.floor(vals.min()), math.ceil(vals.max()), 4)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{y:.1f}" for y in yticks])
                
            
            for row, metric in enumerate(metrics_order):
                ax_ref = axes[row, 0]
                bbox = ax_ref.get_position()
                y_center = 0.5 * (bbox.y0 + bbox.y1)
                
                if metric == 'correlation':
                    text = f'{correlation_type}\n{metric}[-]'
                elif metric == 'PBIAS' : 
                    text = f'{metric}\n[%]'
                elif metric == 'MAE' : 
                    text = f'{metric}\n[mm]'
                elif metric == 'dfs' : 
                    text = f'PET\n[mm]'
                fig.text(0.05, y_center, text,
                         va='center', ha='center', fontsize=14, rotation=90)

    fig.savefig(os.path.join(save_dir, f'{int(EZG_ID)}.png'))
#     plt.tight_layout() 
    plt.show()


    
    
def get_all_stats(dfs):
    '''
    returns annual: mae_matrix, pbias_matrix, corr_matrix, corr_test_type 
            winter: 
            summer:
            spring:
            autumn:
    '''
    results_dict = dict()
    annual_stats = get_stats(dfs)
    winter_dfs = get_seasonalt2ms(dfs, months = [12,1,2]) #djf
    winter_stats = get_stats(winter_dfs)
    summer_dfs = get_seasonalt2ms(dfs, months = [6,7,8]) #jja
    summer_stats = get_stats(summer_dfs)
    spring_djfs = get_seasonalt2ms(dfs, months = [3,4,5]) #mam
    spring_stats = get_stats(spring_djfs)
    autumn_djfs = get_seasonalt2ms(dfs, months = [9,10,11]) #son
    autumn_stats = get_stats(autumn_djfs)
    
    results_dict['all seasons'] = annual_stats
    results_dict['all seasons']['df'] = dfs
    
    results_dict['winter (DJF)'] = winter_stats
    results_dict['winter (DJF)']['df'] = winter_dfs
    
    results_dict['summer (JJA)'] = summer_stats
    results_dict['summer (JJA)']['df'] = summer_dfs
    
    results_dict['spring (MAM)'] = spring_stats
    results_dict['spring (MAM)']['df'] = spring_djfs
    
    results_dict['autumn (SON)'] = autumn_stats
    results_dict['autumn (SON)']['df'] = autumn_djfs
    return results_dict
# results_dict = get_all_stats(dfs)
def fix_hourly(ear5_o, era5l_o, hpet_o, cerra_o):
    ear5, era5l, hpet, cerra = [x.copy(deep = True) for x in [ear5_o, era5l_o, hpet_o, cerra_o]]    
    end = len(cerra)
    ear5 = ear5[1:end]
    era5l = era5l[1:end]
    hpet = hpet[1:end]
    cerra = cerra[1:end]
    return ear5, era5l, hpet, cerra
def get_daily(ear5_o, era5l_o, hpet_o, cerra_o, metref_o):
    param = 't2m'
    ear5, era5l, hpet, cerra, metref = [df.copy(deep = True) 
                                        for df in [ear5_o, era5l_o, hpet_o, cerra_o, metref_o]]
    dfs = [ear5, era5l, hpet, cerra, metref]
    dfs_dt = []
    for df in dfs:
        df = df.copy()
        df["date"] = pd.to_datetime({
            "year":  df["year"].astype(int),
            "month": df["month"].astype(int),
            "day":   df["day"].astype(int)
        })
        df = df.set_index("date")
        dfs_dt.append(df)
    dfs_daily = [df[param].resample("D").sum().to_frame(param) 
                 for df in dfs_dt]
#     print(dfs_daily)
    for i, d in enumerate(dfs_daily):
        d = d.reset_index()
        d["date"] = pd.to_datetime({
            "year":  d["date"].dt.year,
            "month": d["date"].dt.month,
            "day":   d["date"].dt.day
        })
        d["year"]  = d["date"].dt.year
        d["month"] = d["date"].dt.month
        d["day"]   = d["date"].dt.day
        d = d.set_index("date")
        dfs_daily[i] = d
    common_dates = set(dfs_daily[0].index)
    for d in dfs_daily[1:]:
        common_dates &= set(d.index)
    common_dates = sorted(common_dates)
    dfs_aligned = [d.loc[common_dates] for d in dfs_daily]
#     print(dfs_aligned)
    return dfs_aligned
def get_monthly(dfs, param = 't2m'):

    dfs_dt = []
    for df in dfs:
        df = df.copy()
        df["date"] = pd.to_datetime({
            "year":  df["year"].astype(int),
            "month": df["month"].astype(int),
            "day":   df["day"].astype(int)
        })
        df = df.set_index("date")
        dfs_dt.append(df)

    dfs_monthly = [
        df[param].resample("ME").sum().to_frame(param)
        for df in dfs_dt
    ]

    for i, d in enumerate(dfs_monthly):
        d = d.reset_index()
        d["year"]  = d["date"].dt.year
        d["month"] = d["date"].dt.month
        d["day"]   = 1
        d["date"] = pd.to_datetime({
            "year":  d["year"],
            "month": d["month"],
            "day":   d["day"]
        })
        d = d.set_index("date")
        dfs_monthly[i] = d

    common_dates = set(dfs_monthly[0].index)
    for d in dfs_monthly[1:]:
        common_dates &= set(d.index)

    common_dates = sorted(common_dates)

    dfs_aligned = [d.loc[common_dates] for d in dfs_monthly]

    return dfs_aligned

def metric_plots(shapefile_path = r"D:\all catchments\catchments.shp",
                 era5_dir = r'D:\pet\all_pet\ERA5_eto',
                 era5land_dir = r'D:\pet\all_pet\ERA5Land_eto',
                 hpet_dir = r'D:\pet\all_pet\hpet_eto',
                 cerra_dir = r'D:\pet\all_pet\cerra_eto',
                 metref_dir = r'D:\pet\all_pet\metref_eto',
                 mpet_dir = r'D:\pet\all_pet\mpet_eto'):
    '''
    this is the only function where the variables must be declared. 
    shapefile_path: path to shapefile
    all_dirs are the directories where .eto is stored. eto for different dataset should be stored in different dir.
    '''
    dirs = [era5_dir, era5land_dir, hpet_dir,cerra_dir, metref_dir, mpet_dir]
    product_names = ['ERA5', 'ERA5Land', 'hPET','CERRA', 'METREF', 'MPET']
    catchments = gpd.read_file(shapefile_path)
#     all_monthly_results_dict = dict()
#     all_daily_results_dict = dict()
    for num in np.arange(0,catchments.shape[0],1):
        EZG_ID = catchments.Pegel_ID.iloc[num]
        if True:#int(EZG_ID) == 4822106:
#             print('Finding files: ', EZG_ID)
            eto_file_pattern = [os.path.join(dir_path, f'*{int(EZG_ID)}*.eto')
                               for dir_path in dirs]
            files = [glob.glob(file_pattern)
                     for file_pattern in eto_file_pattern]
#             print(files)
            if np.all(np.array([len(_) for _ in files])): #make sure all files exist
#                 print('Reading files')
                print(num, end = '; ')
                ear5, era5l, hpet, cerra, metref, mpet = [prepare_df(_[0]) for _ in files]
                
                ear5, era5l, hpet, cerra = fix_hourly(ear5, era5l, hpet, cerra)
                results_dict = get_all_stats([ear5, era5l, hpet, cerra])
                plot_metrices(results_dict, EZG_ID,labels = ['ERA5', 'ERA5_Land', 'hPET', 'CERRA'],
                              save_dir = r'D:\pet\metrices\hourly' 
                             )
                
                ear5, era5l, hpet, cerra, metref, mpet = [prepare_df(_[0]) for _ in files]
                ear5, era5l, hpet, cerra, metref = get_daily(ear5, era5l, hpet, cerra, metref)
                results_dict = get_all_stats([ear5, era5l, hpet, cerra, metref])
#                 all_daily_results_dict[int(EZG_ID)] = results_dict
                plot_metrices(results_dict, EZG_ID,labels = ['ERA5', 'ERA5_Land', 'hPET', 'CERRA', 'METREF'],
                                              save_dir = r'D:\pet\metrices\daily' 
                                             )
                
                ear5, era5l, hpet, cerra, metref, mpet = [prepare_df(_[0]) for _ in files]
                ear5, era5l, hpet, cerra, metref, mpet = get_monthly([ear5, era5l, hpet, cerra, metref, mpet])
                results_dict = get_all_stats([ear5, era5l, hpet, cerra, metref, mpet])
#                 all_monthly_results_dict[int(EZG_ID)] = results_dict
                plot_metrices(results_dict, EZG_ID ,labels = ['ERA5', 'ERA5_Land', 'hPET', 'CERRA', 'METREF', 'MPET'],
                                              save_dir = r'D:\pet\metrices\monthly' 
                                             )
#                 break
#                 return ear5, era5l, hpet, cerra, metref, mpet
#                 return results_dict
#         if num > 100:
# #             break
#             break
#     return all_daily_results_dict, all_monthly_results_dict
metric_plots()

#### t2m
- PBIAS, MAE, correlation with respect to one another, violin plot for distribution of the datasets
    - hourly (ERA5, ERA5Land, CERRA, HOSTRADA)
- correlation
    - if all datasets normal Pearson
        - else spearman

In [ ]:
from scipy.stats import normaltest
from scipy.stats import spearmanr
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import time
import seaborn as sns
import math
def prepare_df(file_path):
    df = pd.read_csv(file_path, delimiter = '\t', header = None)
    if not 'metref' in file_path.__repr__().lower():
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        return df
    else:
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        df.drop(columns = 'hour', inplace = True)
        return df
def get_pbias(x,y):
    numenor = np.sum(y-x)
    denumenor =  np.sum(np.abs(x)) #np.sum(x)# #done so that if denumenor sum is -ve, it doesn't cancel out underestimation in numenor.
#     denumenor = np.sum(x)
    #     print(numenor ,denumenor)
    return (numenor/denumenor)*100
def get_seasonalt2ms(dfs_o, months = list(range(1,13))):
    dfs = [df.copy(deep = True) for df in dfs_o]
    df_seasons = [df[df['month'].isin(months)] for df in dfs]
    return df_seasons
def get_corrmat(df_seasons): 
    t2ms = np.array([np.array(df.t2m) for df in df_seasons])
    ntest_pvalues = np.array([normaltest(t2m).pvalue for t2m in t2ms]) #scipy default based on kurtosis and skewness 
    if np.all(ntest_pvalues > 0.05):
#         print('All data normal', ntest_pvalues)
        test_type = 'Pearson'
        corr_matrix = np.round(np.corrcoef(t2ms),4)
    else:
#         print('All data non normal', ntest_pvalues)
        test_type = "Spearman"
#         corr_matrix = np.round(spearmanr(t2ms),4)
        corr_matrix = np.round(np.array([[spearmanr(df_ref.t2m, df_mod.t2m).statistic for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    return corr_matrix, test_type
    
def get_stats(df_seasons):
    results = dict()
    mae_matrix = np.round(np.array([[np.mean(np.abs(df_ref.t2m - df_mod.t2m)) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    pbias_matrix = np.round(np.array([[get_pbias(df_ref.t2m, df_mod.t2m) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 2)
    corr_matrix, test_type = get_corrmat(df_seasons)
    results['MAE'] = mae_matrix
    results['PBIAS'] = pbias_matrix
    results['correlation'] = corr_matrix
    results['corr_test'] = test_type
    results['dfs'] = df_seasons
    return results


def plot_metrices(results_dict, EZG_ID,
                 labels = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA'],
                 metrics_order = ["MAE", "PBIAS", "correlation", "dfs"],
                 save_dir = r'D:\temperature_results\metrices'):
    #setting dicts for the plots
    correlation_type = results_dict['all seasons']['corr_test']
    metric_ranges = dict()
    metric_cmaps = {"MAE": 'cividis',#"mako",
                    "PBIAS": "coolwarm", #blue for underestimation, red for over 
                    "correlation": "RdYlGn"} #yl at 0, rd -1, gn +1 
    for metric in metrics_order:
        if metric == 'correlation':
            metric_ranges[metric] = (-1, 1)
        elif metric == 'PBIAS':
            metric_ranges[metric] = (-100, 100)
        elif metric == 'MAE':
            mats = [results_dict[d][metric] for d in list(results_dict.keys())]
            all_vals = np.concatenate([m.flatten() for m in mats])
            metric_ranges[metric] = (all_vals.min()*0.9, all_vals.max()*1.1) #10% addition 
        elif metric == 'dfs':
            mats = np.array([df.t2m.values for df in results_dict['all seasons'][metric]])
            metric_ranges[metric] = (math.floor(mats.min()*0.9), math.ceil(mats.max()*1.1))
    
    #setting up plot 
    nrows=len(metrics_order)
    ncols=len(list(results_dict.keys()))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 16))
    for col, dtype in enumerate(list(results_dict.keys())):
        axes[0, col].set_title(dtype, fontsize=14, pad=30) #name of dtype in each column 

    for col, (dtype, dtype_dict) in enumerate(results_dict.items()):
        for row, metric in enumerate(metrics_order):
            ax = axes[row, col]
            if metric != 'dfs':
                mat = dtype_dict[metric]
                cmap = plt.get_cmap(metric_cmaps[metric]).copy()
                cmap.set_bad(color="lightgrey")
                vmin, vmax = metric_ranges[metric]
                if metric.lower() in ["mae", "correlation"]:
                    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
                    norm = None
                else:
                    mask = None
                    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
                    mask = np.eye(mat.shape[0], dtype=bool)
                mat_ma = np.ma.array(mat, mask=mask) #masked array
                if metric.lower() in ["mae", "pbias"]:
                    fmt=".2f"
                else:
                    fmt=".4f"
                hm = sns.heatmap(mat_ma, 
                                 annot=True, 
                                 fmt=fmt, 
                                 cmap=cmap, 
                                 ax=ax, 
                                 vmin=vmin, 
                                 vmax=vmax,
                                 norm = norm,
                                 cbar=False, 
                                 square=True, 
                                 mask=mask)
            
                if col == ncols - 1:
                    cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])
                    plt.colorbar(hm.collections[0], cax=cax)
                ax.set_xticks([])
#                 if row == nrows - 2:
#                     ax.set_xticks(np.arange(len(labels))+0.5)
#                     ax.set_xticklabels(labels, rotation=45, ha="right")
                ax.set_yticks([])
                if (col == 0) & (row != nrows-1):
                    ax.set_yticks(np.arange(len(labels))+0.5)
                    ax.set_yticklabels(labels, rotation=0)
                if row == 0:#nrows - 2:
                    ax.xaxis.tick_top()
                    ax.xaxis.set_label_position("top")
                    ax.set_xticks(np.arange(len(labels))+0.5)
                    ax.set_xticklabels(labels, rotation=90, ha="left")
                    ax.tick_params(axis="x", bottom=False)
            else:
                dfs = dtype_dict[metric]
                ax.yaxis.grid(True)
                ax.violinplot([df.t2m.values for df in dfs],showmeans=True)#, tick_labels=labels, patch_artist=True)
                ax.set_xticks([y+1 for y in range(len(labels))])
                ax.set_xticklabels(labels, rotation=90, ha="left")
#                 ax.set_ylabel('T [°C]')
#                 ax.set_xticks([])
#                 if col == 0:            
                vals = np.array([df.t2m.values for df in dfs])
                yticks = np.linspace(math.floor(vals.min()), math.ceil(vals.max()), 7)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{y:.0f}" for y in yticks])
                
            
            for row, metric in enumerate(metrics_order):
                ax_ref = axes[row, 0]
                bbox = ax_ref.get_position()
                y_center = 0.5 * (bbox.y0 + bbox.y1)
                
                if metric == 'correlation':
                    text = f'{correlation_type}\n{metric}[-]'
                elif metric == 'PBIAS' : 
                    text = f'{metric}\n[%]'
                elif metric == 'MAE' : 
                    text = f'{metric}\n[°C]'
                elif metric == 'dfs' : 
                    text = f'T\n[°C]'
                fig.text(0.05, y_center, text,
                         va='center', ha='center', fontsize=14, rotation=90)

    fig.savefig(os.path.join(save_dir, f'{int(EZG_ID)}.png'))
#     plt.tight_layout() 
    plt.show()


    
    
def get_all_stats(dfs):
    '''
    returns annual: mae_matrix, pbias_matrix, corr_matrix, corr_test_type 
            winter: 
            summer:
            spring:
            autumn:
    '''
    results_dict = dict()
    annual_stats = get_stats(dfs)
    winter_dfs = get_seasonalt2ms(dfs, months = [12,1,2]) #djf
    winter_stats = get_stats(winter_dfs)
    summer_dfs = get_seasonalt2ms(dfs, months = [6,7,8]) #jja
    summer_stats = get_stats(summer_dfs)
    spring_djfs = get_seasonalt2ms(dfs, months = [3,4,5]) #mam
    spring_stats = get_stats(spring_djfs)
    autumn_djfs = get_seasonalt2ms(dfs, months = [9,10,11]) #son
    autumn_stats = get_stats(autumn_djfs)
    
    results_dict['all seasons'] = annual_stats
    results_dict['all seasons']['df'] = dfs
    
    results_dict['winter (DJF)'] = winter_stats
    results_dict['winter (DJF)']['df'] = winter_dfs
    
    results_dict['summer (JJA)'] = summer_stats
    results_dict['summer (JJA)']['df'] = summer_dfs
    
    results_dict['spring (MAM)'] = spring_stats
    results_dict['spring (MAM)']['df'] = spring_djfs
    
    results_dict['autumn (SON)'] = autumn_stats
    results_dict['autumn (SON)']['df'] = autumn_djfs
    return results_dict
# results_dict = get_all_stats(dfs)
def metric_plots(shapefile_path = r".\catchments.shp",
                   era5_dir = r'.\pet\ERA5\tav' ,
                   hostrada_dir = r'.\pet\HOSTRADA\tav',
                   cerra_dir = r'.\pet\CERRA\tav',
                   era5land_dir = r'.\pet\ERA5_Land\tav'):
    '''
    this is the only function where the variables must be declared. 
    shapefile_path: path to shapefile
    all_dirs are the directories where .eto is stored. eto for different dataset should be stored in different dir.
    '''
    dirs = [era5_dir, era5land_dir, cerra_dir, hostrada_dir]
    product_names = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA']
    catchments = gpd.read_file(shapefile_path)
    for num in np.arange(0,catchments.shape[0],1):
        EZG_ID = catchments.Pegel_ID.iloc[num]
        if True:#int(EZG_ID) == 4822106:
            print('Finding files: ', EZG_ID)
            eto_file_pattern = [os.path.join(dir_path, f'*{int(EZG_ID)}*.tav')
                               for dir_path in dirs]
            files = [glob.glob(file_pattern)
                     for file_pattern in eto_file_pattern]
#             print(files)
            if np.all(np.array([len(_) for _ in files])): #make sure all files exist
                print('Reading files')        
                ear5, era5l, cerra, hostrada = [prepare_df(_[0]) for _ in files]
                results_dict = get_all_stats([ear5, era5l, cerra, hostrada])
                plot_metrices(results_dict, EZG_ID,
                              labels = product_names,
                              save_dir = r'.\pet\summary\metrices\pbias_mod' 
                             )
#                 return results_dict
#         if num > 2:
#             break
#             break
#     return [ear5, era5l, cerra, hostrada]
metric_plots()

## distribution of errors plots
- reuses metric_plot to create dictionary of errors
- the order of reading files is important, no memory of model name, caution here

#### pet

##### metric_plots() modified 

In [ ]:
#### the metric_plots() function is slightly modified to return metric matrices, and not to keep df in dict
from scipy.stats import normaltest
from scipy.stats import spearmanr
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import time
import seaborn as sns
import math
def prepare_df(file_path):
    df = pd.read_csv(file_path, delimiter = '\t', header = None)
    if not 'metref' in file_path.__repr__().lower():
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        return df
    else:
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        df.drop(columns = 'hour', inplace = True)
        return df
def get_pbias(x,y):
    numenor = np.sum(y-x)
#     denumenor =  np.sum(np.abs(x)) #np.sum(x)# #done so that if denumenor sum is -ve, it doesn't cancel out underestimation in numenor.
    denumenor = np.sum(x)
    #     print(numenor ,denumenor)
    return (numenor/denumenor)*100
def get_seasonalt2ms(dfs_o, months = list(range(1,13))):
    dfs = [df.copy(deep = True) for df in dfs_o]
    df_seasons = [df[df['month'].isin(months)] for df in dfs]
    return df_seasons
def get_corrmat(df_seasons): 
    t2ms = np.array([np.array(df.t2m) for df in df_seasons])
    ntest_pvalues = np.array([normaltest(t2m).pvalue for t2m in t2ms]) #scipy default based on kurtosis and skewness 
    if np.all(ntest_pvalues > 0.05):
#         print('All data normal', ntest_pvalues)
        test_type = 'Pearson'
        corr_matrix = np.round(np.corrcoef(t2ms),4)
    else:
#         print('All data non normal', ntest_pvalues)
        test_type = "Spearman"
#         corr_matrix = np.round(spearmanr(t2ms),4)
        corr_matrix = np.round(np.array([[spearmanr(df_ref.t2m, df_mod.t2m).statistic for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    return corr_matrix, test_type
    
def get_stats(df_seasons):
    results = dict()
    mae_matrix = np.round(np.array([[np.mean(np.abs(df_ref.t2m - df_mod.t2m)) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    pbias_matrix = np.round(np.array([[get_pbias(df_ref.t2m, df_mod.t2m) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 2)
    corr_matrix, test_type = get_corrmat(df_seasons)
    results['MAE'] = mae_matrix
    results['PBIAS'] = pbias_matrix
    results['correlation'] = corr_matrix
    results['corr_test'] = test_type
#     results['dfs'] = df_seasons
    return results


def plot_metrices(results_dict, EZG_ID,
                 labels = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA'],
                 metrics_order = ["MAE", "PBIAS", "correlation", "dfs"],
                 save_dir = r'D:\temperature_results\metrices'):
    #setting dicts for the plots
    correlation_type = results_dict['all seasons']['corr_test']
    metric_ranges = dict()
    metric_cmaps = {"MAE": 'cividis',#"mako",
                    "PBIAS": "coolwarm", #blue for underestimation, red for over 
                    "correlation": "RdYlGn"} #yl at 0, rd -1, gn +1 
    for metric in metrics_order:
        if metric == 'correlation':
            metric_ranges[metric] = (-1, 1)
        elif metric == 'PBIAS':
            metric_ranges[metric] = (-100, 100)
        elif metric == 'MAE':
            mats = [results_dict[d][metric] for d in list(results_dict.keys())]
            all_vals = np.concatenate([m.flatten() for m in mats])
            metric_ranges[metric] = (all_vals.min()*0.9, all_vals.max()*1.1) #10% addition 
        elif metric == 'dfs':
            mats = np.array([df.t2m.values for df in results_dict['all seasons'][metric]])
            metric_ranges[metric] = (math.floor(mats.min()*0.9), math.ceil(mats.max()*1.1))
    
    #setting up plot 
    nrows=len(metrics_order)
    ncols=len(list(results_dict.keys()))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(22, 20))
    for col, dtype in enumerate(list(results_dict.keys())):
        axes[0, col].set_title(dtype, fontsize=14, pad=30) #name of dtype in each column 

    for col, (dtype, dtype_dict) in enumerate(results_dict.items()):
        for row, metric in enumerate(metrics_order):
            ax = axes[row, col]
            if metric != 'dfs':
                mat = dtype_dict[metric]
                cmap = plt.get_cmap(metric_cmaps[metric]).copy()
                cmap.set_bad(color="lightgrey")
                vmin, vmax = metric_ranges[metric]
                if metric.lower() in ["mae", "correlation"]:
                    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
                    norm = None
                else:
                    mask = None
                    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
                    mask = np.eye(mat.shape[0], dtype=bool)
                mat_ma = np.ma.array(mat, mask=mask) #masked array
                if metric.lower() in ["mae", "pbias"]:
                    fmt=".2f"
                else:
                    fmt=".2f"
                hm = sns.heatmap(mat_ma, 
                                 annot=True, 
                                 fmt=fmt, 
                                 cmap=cmap, 
                                 ax=ax, 
                                 vmin=vmin, 
                                 vmax=vmax,
                                 norm = norm,
                                 cbar=False, 
                                 square=True, 
                                 mask=mask)
            
                if col == ncols - 1:
                    cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])
                    plt.colorbar(hm.collections[0], cax=cax)
                ax.set_xticks([])
#                 if row == nrows - 2:
#                     ax.set_xticks(np.arange(len(labels))+0.5)
#                     ax.set_xticklabels(labels, rotation=45, ha="right")
                ax.set_yticks([])
                if (col == 0) & (row != nrows-1):
                    ax.set_yticks(np.arange(len(labels))+0.5)
                    ax.set_yticklabels(labels, rotation=0)
                if row == 0:#nrows - 2:
                    ax.xaxis.tick_top()
                    ax.xaxis.set_label_position("top")
                    ax.set_xticks(np.arange(len(labels))+0.5)
                    ax.set_xticklabels(labels, rotation=90, ha="left")
                    ax.tick_params(axis="x", bottom=False)
            else:
                dfs = dtype_dict[metric]
                ax.yaxis.grid(True)
                ax.violinplot([df.t2m.values for df in dfs],showmeans=True)#, tick_labels=labels, patch_artist=True)
                ax.set_xticks([y+1 for y in range(len(labels))])
                ax.set_xticklabels(labels, rotation=90, ha="left")
#                 ax.set_ylabel('T [°C]')
#                 ax.set_xticks([])
#                 if col == 0:            
                vals = np.array([df.t2m.values for df in dfs])
                yticks = np.linspace(math.floor(vals.min()), math.ceil(vals.max()), 4)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{y:.1f}" for y in yticks])
                
            
            for row, metric in enumerate(metrics_order):
                ax_ref = axes[row, 0]
                bbox = ax_ref.get_position()
                y_center = 0.5 * (bbox.y0 + bbox.y1)
                
                if metric == 'correlation':
                    text = f'{correlation_type}\n{metric}[-]'
                elif metric == 'PBIAS' : 
                    text = f'{metric}\n[%]'
                elif metric == 'MAE' : 
                    text = f'{metric}\n[mm]'
                elif metric == 'dfs' : 
                    text = f'PET\n[mm]'
                fig.text(0.05, y_center, text,
                         va='center', ha='center', fontsize=14, rotation=90)

    fig.savefig(os.path.join(save_dir, f'{int(EZG_ID)}.png'))
#     plt.tight_layout() 
    plt.show()


    
    
def get_all_stats(dfs):
    '''
    returns annual: mae_matrix, pbias_matrix, corr_matrix, corr_test_type 
            winter: 
            summer:
            spring:
            autumn:
    '''
    results_dict = dict()
    annual_stats = get_stats(dfs)
    winter_dfs = get_seasonalt2ms(dfs, months = [12,1,2]) #djf
    winter_stats = get_stats(winter_dfs)
    summer_dfs = get_seasonalt2ms(dfs, months = [6,7,8]) #jja
    summer_stats = get_stats(summer_dfs)
    spring_djfs = get_seasonalt2ms(dfs, months = [3,4,5]) #mam
    spring_stats = get_stats(spring_djfs)
    autumn_djfs = get_seasonalt2ms(dfs, months = [9,10,11]) #son
    autumn_stats = get_stats(autumn_djfs)
    
    results_dict['all seasons'] = annual_stats
#     results_dict['all seasons']['df'] = dfs
    
    results_dict['winter (DJF)'] = winter_stats
#     results_dict['winter (DJF)']['df'] = winter_dfs
    
    results_dict['summer (JJA)'] = summer_stats
#     results_dict['summer (JJA)']['df'] = summer_dfs
    
    results_dict['spring (MAM)'] = spring_stats
#     results_dict['spring (MAM)']['df'] = spring_djfs
    
    results_dict['autumn (SON)'] = autumn_stats
#     results_dict['autumn (SON)']['df'] = autumn_djfs
    return results_dict
# results_dict = get_all_stats(dfs)
def fix_hourly(ear5_o, era5l_o, hpet_o, cerra_o):
    ear5, era5l, hpet, cerra = [x.copy(deep = True) for x in [ear5_o, era5l_o, hpet_o, cerra_o]]    
    end = len(cerra)
    ear5 = ear5[1:end]
    era5l = era5l[1:end]
    hpet = hpet[1:end]
    cerra = cerra[1:end]
    return ear5, era5l, hpet, cerra
def get_daily(ear5_o, era5l_o, hpet_o, cerra_o, metref_o):
    param = 't2m'
    ear5, era5l, hpet, cerra, metref = [df.copy(deep = True) 
                                        for df in [ear5_o, era5l_o, hpet_o, cerra_o, metref_o]]
    dfs = [ear5, era5l, hpet, cerra, metref]
    dfs_dt = []
    for df in dfs:
        df = df.copy()
        df["date"] = pd.to_datetime({
            "year":  df["year"].astype(int),
            "month": df["month"].astype(int),
            "day":   df["day"].astype(int)
        })
        df = df.set_index("date")
        dfs_dt.append(df)
    dfs_daily = [df[param].resample("D").sum().to_frame(param) 
                 for df in dfs_dt]
#     print(dfs_daily)
    for i, d in enumerate(dfs_daily):
        d = d.reset_index()
        d["date"] = pd.to_datetime({
            "year":  d["date"].dt.year,
            "month": d["date"].dt.month,
            "day":   d["date"].dt.day
        })
        d["year"]  = d["date"].dt.year
        d["month"] = d["date"].dt.month
        d["day"]   = d["date"].dt.day
        d = d.set_index("date")
        dfs_daily[i] = d
    common_dates = set(dfs_daily[0].index)
    for d in dfs_daily[1:]:
        common_dates &= set(d.index)
    common_dates = sorted(common_dates)
    dfs_aligned = [d.loc[common_dates] for d in dfs_daily]
#     print(dfs_aligned)
    return dfs_aligned
def get_monthly(dfs, param = 't2m'):

    dfs_dt = []
    for df in dfs:
        df = df.copy()
        df["date"] = pd.to_datetime({
            "year":  df["year"].astype(int),
            "month": df["month"].astype(int),
            "day":   df["day"].astype(int)
        })
        df = df.set_index("date")
        dfs_dt.append(df)

    dfs_monthly = [
        df[param].resample("ME").sum().to_frame(param)
        for df in dfs_dt
    ]

    for i, d in enumerate(dfs_monthly):
        d = d.reset_index()
        d["year"]  = d["date"].dt.year
        d["month"] = d["date"].dt.month
        d["day"]   = 1
        d["date"] = pd.to_datetime({
            "year":  d["year"],
            "month": d["month"],
            "day":   d["day"]
        })
        d = d.set_index("date")
        dfs_monthly[i] = d

    common_dates = set(dfs_monthly[0].index)
    for d in dfs_monthly[1:]:
        common_dates &= set(d.index)

    common_dates = sorted(common_dates)

    dfs_aligned = [d.loc[common_dates] for d in dfs_monthly]

    return dfs_aligned

def metric_plots(shapefile_path = r".\catchments.shp",
                 era5_dir = r'.\pet\all_pet\ERA5_eto',
                 era5land_dir = r'.\pet\all_pet\ERA5Land_eto',
                 hpet_dir = r'.\pet\all_pet\hpet_eto',
                 cerra_dir = r'.\pet\all_pet\cerra_eto',
                 metref_dir = r'.\pet\all_pet\metref_eto',
                 mpet_dir = r'.\pet\all_pet\mpet_eto'):
    '''
    this is the only function where the variables must be declared. 
    shapefile_path: path to shapefile
    all_dirs are the directories where .eto is stored. eto for different dataset should be stored in different dir.
    '''
    dirs = [era5_dir, era5land_dir, hpet_dir,cerra_dir, metref_dir, mpet_dir]
    product_names = ['ERA5', 'ERA5Land', 'hPET','CERRA', 'METREF', 'MPET']
    catchments = gpd.read_file(shapefile_path)
    all_monthly_results_dict = dict()
    all_daily_results_dict = dict()
    for num in np.arange(0,catchments.shape[0],1):
        EZG_ID = catchments.Pegel_ID.iloc[num]
        if True:#int(EZG_ID) == 4822106:
#             print('Finding files: ', EZG_ID)
            eto_file_pattern = [os.path.join(dir_path, f'*{int(EZG_ID)}*.eto')
                               for dir_path in dirs]
            files = [glob.glob(file_pattern)
                     for file_pattern in eto_file_pattern]
#             print(files)
            if np.all(np.array([len(_) for _ in files])): #make sure all files exist
#                 print('Reading files')
                print(num, end = '; ')
                ear5, era5l, hpet, cerra, metref, mpet = [prepare_df(_[0]) for _ in files]
                
#                 ear5, era5l, hpet, cerra = fix_hourly(ear5, era5l, hpet, cerra)
#                 results_dict = get_all_stats([ear5, era5l, hpet, cerra])
#                 plot_metrices(results_dict, EZG_ID,labels = ['ERA5', 'ERA5_Land', 'hPET', 'CERRA'],
#                               save_dir = r'D:\pet\metrices\hourly' 
#                              )
                
                ear5, era5l, hpet, cerra, metref, mpet = [prepare_df(_[0]) for _ in files]
                ear5, era5l, hpet, cerra, metref = get_daily(ear5, era5l, hpet, cerra, metref)
                results_dict = get_all_stats([ear5, era5l, hpet, cerra, metref])
                all_daily_results_dict[int(EZG_ID)] = results_dict
#                 plot_metrices(results_dict, EZG_ID,labels = ['ERA5', 'ERA5_Land', 'hPET', 'CERRA', 'METREF'],
#                                               save_dir = r'D:\pet\metrices\daily' 
#                                              )
                
                ear5, era5l, hpet, cerra, metref, mpet = [prepare_df(_[0]) for _ in files]
                ear5, era5l, hpet, cerra, metref, mpet = get_monthly([ear5, era5l, hpet, cerra, metref, mpet])
                results_dict = get_all_stats([ear5, era5l, hpet, cerra, metref, mpet])
                all_monthly_results_dict[int(EZG_ID)] = results_dict
#                 plot_metrices(results_dict, EZG_ID ,labels = ['ERA5', 'ERA5_Land', 'hPET', 'CERRA', 'METREF', 'MPET'],
#                                               save_dir = r'D:\pet\metrices\monthly' 
#                                              )
#                 break
#                 return ear5, era5l, hpet, cerra, metref, mpet
#                 return results_dict
#         if num > 100:
# #             break
#             break
    return all_daily_results_dict, all_monthly_results_dict
%time daily_dict, monthly_dict = metric_plots()



#dict saved as .txt file for reuse
import pickle
with open(r".\pet\metrices\daily_dict.txt", "wb") as f:
    pickle.dump(daily_dict, f)
with open(r".\pet\metrices\monthly_dict.txt", "wb") as f:
    pickle.dump(monthly_dict, f)

#dict can be reread as 
with open(r".\pet\metrices\daily_dict.txt", "rb") as f:
    daily_dict = pickle.load(f)
with open(r".\pet\metrices\monthly_dict.txt", "rb") as f:
    monthly_dict = pickle.load(f)

##### functions to get  error distribution versus catchment attributes

In [ ]:
import geopandas as gpd 
def get_reference_results_only(ref_dict, 
                               shapefile_path = r'D:\all catchments\catchments_all.shp'):
    '''
    From the dictionary of results obtained by slightly modifying metric_plots, errors of models with respect to reference catchment is taken. 
    Reference catchment always read at last. 
    ERA5Land can be skipped for pet datasets.
    
    Also creates a dictionary of catchment attributes {'cat_id': {'slope_avrg': , 'z_avrg': }}
    '''
    catchments = gpd.read_file(shapefile_path)
    cat_result = dict()
    cat_attrs  = dict()
    for cat_id in ref_dict.keys():
        cat_result[cat_id] = dict()
        cat_attrs[cat_id] = dict()
        cat_area = catchments[catchments['Pegel_ID'] == np.float64(cat_id)].Area_km2.item()
        cat_attrs[cat_id]['z_avrg'] = catchments[catchments['Pegel_ID'] == np.float64(cat_id)].z_avrg.item()
        cat_attrs[cat_id]['slope_avrg'] = catchments[catchments['Pegel_ID'] == np.float64(cat_id)].slope_avrg.item()
    #     print(type(cat_area),cat_area)
        for accum_window in ref_dict[cat_id].keys():
            cat_result[cat_id][accum_window] = dict()
            for metric in ['MAE', 'PBIAS', 'correlation']:
                cat_result[cat_id][accum_window][metric] = dict()
                if metric == 'MAE': #mae normalized by area as PET --> Area #mae already normalized during skimage reduction by = nanmean
                    cat_result[cat_id][accum_window]['MAE'] = ref_dict[cat_id][accum_window][metric][-1,:]#/cat_area
                else:
                    cat_result[cat_id][accum_window][metric] = ref_dict[cat_id][accum_window][metric][-1,:]
    return cat_result, cat_attrs


def plot_ref_results(cat_results, cat_attrs, 
                     metric = 'MAE', metric_unit = 'mm', 
                     models = ['ERA5', 'ERA5Land', 'hPET','CERRA', 'METREF']):
    z_avrg = np.array([cat_attrs[cat_id]['z_avrg'] for cat_id in cat_attrs.keys()])
    slope = np.array([cat_attrs[cat_id]['slope_avrg'] for cat_id in cat_attrs.keys()])
    
    fig, axes = plt.subplots(
        nrows=2, ncols=5,
        figsize=(24, 18),
        sharex=False, sharey=False)
    season_names = ['all seasons', 'winter (DJF)', 'summer (JJA)', 'spring (MAM)', 'autumn (SON)']
    row_names = ["z avrg [m]", "slope avrg [%]"]
    for j, name in enumerate(season_names):
        axes[0, j].set_title(name, fontsize=14)
    for i, name in enumerate(row_names):
        axes[i, 0].set_ylabel(f'{metric} [{metric_unit}]', fontsize=12, rotation=90, labelpad=20)
        for j in range(len(season_names)):
            axes[i, j].set_xlabel(name, fontsize=12, rotation=0, labelpad=20)
    
    for i in range(len(row_names)):
        if i == 0:
            x_data = z_avrg
        if i == 1:
            x_data = slope
        for j,season in enumerate(season_names):
            metric_season_array = np.array([cat_results[cat_id][season][metric] for cat_id in cat_results.keys()])
            for k, model in enumerate(models):
#                 if model != 'ERA5Land':
                if True:
                    res = metric_season_array[:,k]
                    corr = np.corrcoef(x_data, res)[0,1]
                    axes[i,j].scatter(x_data, res, label  = f'{model}, r = {corr:.2f}')
#                 axes[i,j].scatter(res, x_data, label  = f'{model}, r = {corr:.2f}')
            axes[i,j].legend(loc="upper right")
#     plt.suptitle(metric)
    plt.tight_layout()
    plt.show()

#for monthly accumulation, mPET taken as reference
cat_results, cat_attrs = get_reference_results_only(monthly_dict)
plot_ref_results(cat_results, cat_attrs,
                 metric = 'MAE', metric_unit = 'mm', models = ['ERA5', 'ERA5Land', 'hPET','CERRA'])
plot_ref_results(cat_results, cat_attrs,
                 metric = 'PBIAS', metric_unit = '%', models = ['ERA5', 'ERA5Land', 'hPET','CERRA'])
plot_ref_results(cat_results, cat_attrs,
                 metric = 'correlation', metric_unit = '-', models = ['ERA5', 'ERA5Land', 'hPET','CERRA'])



#for daily accumulation, metref taken as reference
cat_results, cat_attrs = get_reference_results_only(daily_dict)
plot_ref_results(cat_results, cat_attrs,
                 metric = 'MAE', metric_unit = 'mm', models = ['ERA5', 'ERA5Land', 'hPET','CERRA'])
plot_ref_results(cat_results, cat_attrs,
                 metric = 'PBIAS', metric_unit = '%', models = ['ERA5', 'ERA5Land', 'hPET','CERRA'])

cat_results, cat_attrs = get_reference_results_only(daily_dict)
plot_ref_results(cat_results, cat_attrs, 
                 metric = 'MAE', season = 'summer (JJA)',
                 models = ['ERA5', 'ERA5Land', 'hPET','CERRA'])

#### t2m

##### modified plot_metrices() func 

In [ ]:
from scipy.stats import normaltest
from scipy.stats import spearmanr
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import time
import seaborn as sns
import math
def prepare_df(file_path):
    df = pd.read_csv(file_path, delimiter = '\t', header = None)
    if not 'metref' in file_path.__repr__().lower():
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        return df
    else:
        df.columns = ['year', 'month', 'day', 'hour', 't2m']
        df.drop(columns = 'hour', inplace = True)
        return df
def get_pbias(x,y):
    numenor = np.sum(y-x)
    denumenor =  np.sum(np.abs(x)) #np.sum(x)# #done so that if denumenor sum is -ve, it doesn't cancel out underestimation in numenor.
#     denumenor = np.sum(x)
    #     print(numenor ,denumenor)
    return (numenor/denumenor)*100
def get_seasonalt2ms(dfs_o, months = list(range(1,13))):
    dfs = [df.copy(deep = True) for df in dfs_o]
    df_seasons = [df[df['month'].isin(months)] for df in dfs]
    return df_seasons
def get_corrmat(df_seasons): 
    t2ms = np.array([np.array(df.t2m) for df in df_seasons])
    ntest_pvalues = np.array([normaltest(t2m).pvalue for t2m in t2ms]) #scipy default based on kurtosis and skewness 
    if np.all(ntest_pvalues > 0.05):
#         print('All data normal', ntest_pvalues)
        test_type = 'Pearson'
        corr_matrix = np.round(np.corrcoef(t2ms),4)
    else:
#         print('All data non normal', ntest_pvalues)
        test_type = "Spearman"
#         corr_matrix = np.round(spearmanr(t2ms),4)
        corr_matrix = np.round(np.array([[spearmanr(df_ref.t2m, df_mod.t2m).statistic for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    return corr_matrix, test_type
    
def get_stats(df_seasons):
    results = dict()
    mae_matrix = np.round(np.array([[np.mean(np.abs(df_ref.t2m - df_mod.t2m)) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 4)
    pbias_matrix = np.round(np.array([[get_pbias(df_ref.t2m, df_mod.t2m) for df_mod in df_seasons] 
                                    for df_ref in df_seasons]), 2)
    corr_matrix, test_type = get_corrmat(df_seasons)
    results['MAE'] = mae_matrix
    results['PBIAS'] = pbias_matrix
    results['correlation'] = corr_matrix
    results['corr_test'] = test_type
    #results['dfs'] = df_seasons
    return results


def plot_metrices(results_dict, EZG_ID,
                 labels = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA'],
                 metrics_order = ["MAE", "PBIAS", "correlation", "dfs"],
                 save_dir = r'D:\temperature_results\metrices'):
    #setting dicts for the plots
    correlation_type = results_dict['all seasons']['corr_test']
    metric_ranges = dict()
    metric_cmaps = {"MAE": 'cividis',#"mako",
                    "PBIAS": "coolwarm", #blue for underestimation, red for over 
                    "correlation": "RdYlGn"} #yl at 0, rd -1, gn +1 
    for metric in metrics_order:
        if metric == 'correlation':
            metric_ranges[metric] = (-1, 1)
        elif metric == 'PBIAS':
            metric_ranges[metric] = (-100, 100)
        elif metric == 'MAE':
            mats = [results_dict[d][metric] for d in list(results_dict.keys())]
            all_vals = np.concatenate([m.flatten() for m in mats])
            metric_ranges[metric] = (all_vals.min()*0.9, all_vals.max()*1.1) #10% addition 
        elif metric == 'dfs':
            mats = np.array([df.t2m.values for df in results_dict['all seasons'][metric]])
            metric_ranges[metric] = (math.floor(mats.min()*0.9), math.ceil(mats.max()*1.1))
    
    #setting up plot 
    nrows=len(metrics_order)
    ncols=len(list(results_dict.keys()))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 16))
    for col, dtype in enumerate(list(results_dict.keys())):
        axes[0, col].set_title(dtype, fontsize=14, pad=30) #name of dtype in each column 

    for col, (dtype, dtype_dict) in enumerate(results_dict.items()):
        for row, metric in enumerate(metrics_order):
            ax = axes[row, col]
            if metric != 'dfs':
                mat = dtype_dict[metric]
                cmap = plt.get_cmap(metric_cmaps[metric]).copy()
                cmap.set_bad(color="lightgrey")
                vmin, vmax = metric_ranges[metric]
                if metric.lower() in ["mae", "correlation"]:
                    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
                    norm = None
                else:
                    mask = None
                    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
                    mask = np.eye(mat.shape[0], dtype=bool)
                mat_ma = np.ma.array(mat, mask=mask) #masked array
                if metric.lower() in ["mae", "pbias"]:
                    fmt=".2f"
                else:
                    fmt=".4f"
                hm = sns.heatmap(mat_ma, 
                                 annot=True, 
                                 fmt=fmt, 
                                 cmap=cmap, 
                                 ax=ax, 
                                 vmin=vmin, 
                                 vmax=vmax,
                                 norm = norm,
                                 cbar=False, 
                                 square=True, 
                                 mask=mask)
            
                if col == ncols - 1:
                    cax = ax.inset_axes([1.02, 0.1, 0.04, 0.8])
                    plt.colorbar(hm.collections[0], cax=cax)
                ax.set_xticks([])
#                 if row == nrows - 2:
#                     ax.set_xticks(np.arange(len(labels))+0.5)
#                     ax.set_xticklabels(labels, rotation=45, ha="right")
                ax.set_yticks([])
                if (col == 0) & (row != nrows-1):
                    ax.set_yticks(np.arange(len(labels))+0.5)
                    ax.set_yticklabels(labels, rotation=0)
                if row == 0:#nrows - 2:
                    ax.xaxis.tick_top()
                    ax.xaxis.set_label_position("top")
                    ax.set_xticks(np.arange(len(labels))+0.5)
                    ax.set_xticklabels(labels, rotation=90, ha="left")
                    ax.tick_params(axis="x", bottom=False)
            else:
                dfs = dtype_dict[metric]
                ax.yaxis.grid(True)
                ax.violinplot([df.t2m.values for df in dfs],showmeans=True)#, tick_labels=labels, patch_artist=True)
                ax.set_xticks([y+1 for y in range(len(labels))])
                ax.set_xticklabels(labels, rotation=90, ha="left")
#                 ax.set_ylabel('T [°C]')
#                 ax.set_xticks([])
#                 if col == 0:            
                vals = np.array([df.t2m.values for df in dfs])
                yticks = np.linspace(math.floor(vals.min()), math.ceil(vals.max()), 7)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{y:.0f}" for y in yticks])
                
            
            for row, metric in enumerate(metrics_order):
                ax_ref = axes[row, 0]
                bbox = ax_ref.get_position()
                y_center = 0.5 * (bbox.y0 + bbox.y1)
                
                if metric == 'correlation':
                    text = f'{correlation_type}\n{metric}[-]'
                elif metric == 'PBIAS' : 
                    text = f'{metric}\n[%]'
                elif metric == 'MAE' : 
                    text = f'{metric}\n[°C]'
                elif metric == 'dfs' : 
                    text = f'T\n[°C]'
                fig.text(0.05, y_center, text,
                         va='center', ha='center', fontsize=14, rotation=90)

    fig.savefig(os.path.join(save_dir, f'{int(EZG_ID)}.png'))
#     plt.tight_layout() 
    plt.show()


    
    
def get_all_stats(dfs):
    '''
    returns annual: mae_matrix, pbias_matrix, corr_matrix, corr_test_type 
            winter: 
            summer:
            spring:
            autumn:
    '''
    results_dict = dict()
    annual_stats = get_stats(dfs)
    winter_dfs = get_seasonalt2ms(dfs, months = [12,1,2]) #djf
    winter_stats = get_stats(winter_dfs)
    summer_dfs = get_seasonalt2ms(dfs, months = [6,7,8]) #jja
    summer_stats = get_stats(summer_dfs)
    spring_djfs = get_seasonalt2ms(dfs, months = [3,4,5]) #mam
    spring_stats = get_stats(spring_djfs)
    autumn_djfs = get_seasonalt2ms(dfs, months = [9,10,11]) #son
    autumn_stats = get_stats(autumn_djfs)
    
    results_dict['all seasons'] = annual_stats
#     results_dict['all seasons']['df'] = dfs
    
    results_dict['winter (DJF)'] = winter_stats
#     results_dict['winter (DJF)']['df'] = winter_dfs
    
    results_dict['summer (JJA)'] = summer_stats
#     results_dict['summer (JJA)']['df'] = summer_dfs
    
    results_dict['spring (MAM)'] = spring_stats
#     results_dict['spring (MAM)']['df'] = spring_djfs
    
    results_dict['autumn (SON)'] = autumn_stats
#     results_dict['autumn (SON)']['df'] = autumn_djfs
    return results_dict
# results_dict = get_all_stats(dfs)
def metric_plots(shapefile_path = r"D:\all catchments\catchments.shp",
                   era5_dir = r'D:\era5\era5_t2m\tav' ,
                   hostrada_dir = r'D:\hostrada\tav',
                   cerra_dir = r'D:\cerra_nct2m\2_cerra_annual_ger\tav',
                   era5land_dir = r'D:\era5_land\era5_landt2m\results\tav'):
    '''
    this is the only function where the variables must be declared. 
    shapefile_path: path to shapefile
    all_dirs are the directories where .eto is stored. eto for different dataset should be stored in different dir.
    '''
    dirs = [era5_dir, era5land_dir, cerra_dir, hostrada_dir]
    product_names = ['ERA5', 'ERA5Land', 'CERRA', 'HOSTRADA']
    catchments = gpd.read_file(shapefile_path)
    all_hourly_results_dict = dict()
    for num in np.arange(0,catchments.shape[0],1):
        EZG_ID = catchments.Pegel_ID.iloc[num]
        if True:#int(EZG_ID) == 4822106:
#             print('Finding files: ', EZG_ID)
            eto_file_pattern = [os.path.join(dir_path, f'*{int(EZG_ID)}*.tav')
                               for dir_path in dirs]
            files = [glob.glob(file_pattern)
                     for file_pattern in eto_file_pattern]
#             print(files)
            if np.all(np.array([len(_) for _ in files])): #make sure all files exist
#                 print('Reading files')
                print(num, end = '; ')
                ear5, era5l, cerra, hostrada = [prepare_df(_[0]) for _ in files]
                results_dict = get_all_stats([ear5, era5l, cerra, hostrada])
#                 plot_metrices(results_dict, EZG_ID,
#                               labels = product_names,
#                               save_dir = r'D:\temperature_results\metrices\pbias_mod' 
#                              )
                all_hourly_results_dict[int(EZG_ID)] = results_dict
#                 return results_dict
#         if num > 40:
#             break
#             break
#     return [ear5, era5l, cerra, hostrada]
    return all_hourly_results_dict
hourly_dict = metric_plots()


#dict saved as this func takes about 12 minutes for one complete loop 
import pickle
with open(r"D:\pet\metrices\t2m_hourly_dict.txt", "wb") as f:
    pickle.dump(hourly_dict, f)

#dict can be reread as 
with open("D:\pet\metrices\t2m_hourly_dict.txt", "rb") as f:
    hourly_dict = pickle.load(f)

##### same function as PET used  get error distribution plots 

In [ ]:
cat_results, cat_attrs = get_reference_results_only(hourly_dict)
plot_ref_results(cat_results, cat_attrs,
                 metric = 'correlation', metric_unit = '-', models = ['ERA5', 'ERA5Land','CERRA'])